In [1]:
!pip install monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 7.0 MB/s eta 0:00:00


In [2]:
# %% [markdown CELL 1] ========================================================
# N1 - train the control and the topology arm (GPU session)
#
# Train.  GPU session, 9 h cap, budget-guarded.
#
# Two arms by default (see RUN_QUEUE): 'base' = Dice + distance-weighted BCE (the control)
# and 'sr50' = the same plus a skeleton-recall term with a learnable weight that starts at
# ZERO, so the arm is bit-identical to its control at step 0.  Finished runs leave a DONE
# marker: re-commit, re-attach the output as ADOPT_FROM, and the queue skips them.

# %% [CELL 2] SETTINGS -------------------------------------------------------
RUN_QUEUE = [
    # (arm, seed).  'base' = Dice + distance-weighted BCE (the control).
    # 'sr25'/'sr50'/'sr100' = the same + structure recall (F. in the library: recall of the
    #   prediction on the per-component centre-line of the ground truth) with a learnable
    #   weight whose
    # L2 target is the number after 'sr' (0.25 / 0.5 / 1.0), initialised at zero.
    ("base", 42),
    ("sr50", 42),
    # add these two when you want the seed spread (the honest error bar on every number):
    # ("base", 43), ("sr50", 43),
]
W_TOPOLOGY        = 0.50   # nominal strength of the structure-recall term in the 'sr' arm
W_TOPOLOGY_WARMUP = 400    # iterations over which it ramps 0 -> nominal.  Step 0 is
                           # therefore bit-identical to the control, and the term has to EARN
                           # its place on the held-out metrics (N2/N3) -- which is the only
                           # place that question can be answered (see TopologyLoss: a LEARNABLE
                           # multiplier provably drives itself to zero instead)
TIME_BUDGET_H   = 7.2      # leave >1 h of the 9 h session for setup + validation
TARGET_ITERS    = 24000     # what you would run if time were free
MAX_ITERS_CAP   = 24000    # never train longer than tis
RUN_TARGET_MIN  = 180       # per-run wall clock the PILOT cell aims for.  A run that finishes
                           # early is NORMAL only if it reached TARGET_ITERS; the PILOT prints
                           # how many iterations this session adds.
MIN_RUN_MIN     = 20       # do not start a run with less than this left in the session
GRAD_CLIP   = 5.0          # clip the gradient norm (both arms, identical).  A single
                           # large-gradient batch is the most likely way an fp16 backward writes
                           # inf/nan into fp32 weights -- which is what killed the sr50 run at
                           # iteration 1103 while the base arm survived 8000.
MAX_RECOVER = 3            # a run may roll back to its last checkpoint and carry on with half the
                           # LR this many times instead of dying: a 2-hour session should not be
                           # lost to one bad batch.  Set 0 to make any non-finite loss fatal.
IGNORE_FAILED_RUNS = True  # a finished run whose validation score sits at the "predicts
                           # nothing" level (<= FAILED_BEST) is treated as NOT run: its stale
                           # checkpoints are moved aside (*.failed) and it is retrained.  Without
                           # this, a NaN-poisoned run from an older build is silently SKIPPED and
                           # its dead checkpoint goes on to N2.  Set False to keep such a run.
FAILED_BEST     = 0.08     # the empty-prediction signature is 2/(12*3) = 0.056 on the default
                           # VAL_SUBSET=12; anything under this is not a real result
EXTEND          = True     # True = a run that is already DONE is CONTINUED for another
                           # RUN_TARGET_MIN batch from last.pt instead of being skipped.  Use it
                           # when history.json shows the validation curve still rising at the end
AMP             = None     # fp16 autocast: None = on for CUDA, off for CPU; True/False forces it.
                           # The FORWARD runs in fp16 for speed, but the LOSS is always computed in
                           # fp32: MONAI's Dice loss sums p^2 over a whole patch, and in fp16 that
                           # sum overflows (1.8e5 > 65504) -> inf -> nan loss -> nan weights -> the
                           # model predicts nothing for the rest of the run.  Leave this None.
PILOT_WARMUP    = 3        # PILOT iterations discarded before timing: CUDA/cuDNN warm-up
PILOT_TIMED     = 12       # plus the first cold reads of the mounted dataset.  Timing those made
                           # the iteration budget ~8x too small on a real run -- the "45 min" run
                           # finished in 4.6 min with 1/8 of the iterations, and never learned
PATCH       = (96, 96, 96)
BATCH       = 2
NUM_WORKERS = 2
LR          = 2e-4
WEIGHT_DECAY = 1e-5
VAL_SUBSET  = 12           # cases used for the in-training validation (fast)
VAL_EVERY_ROUNDS = 8       # validation rounds inside a run
CKPT_EVERY  = 500
SEED        = 42
WORK        = "/kaggle/working"
INPUT_ROOT  = "/kaggle/input"
CACHE_BASE  = None         # None = auto-discover every attached "cache_base*" (shards ok)
SKEL_ROOT   = None         # None = auto-discover every attached "skel" dataset
ADOPT_FROM  = None         # e.g. "/kaggle/input/brats-runs-v1" to resume previous runs

# %% [CELL 3] SHARED LIBRARY (bratsx + safepp) -- identical in every notebook -----
from __future__ import annotations

# ============================================================================
# bratsx.py -- shared library for the BraTS2020 x fractional-DoD study
# (attached to each notebook as a single "SHARED LIBRARY" cell)
#
# Contents
#   A. fractional operators  : riesz_weights, RieszConv3d, FractionalOrderMixture,
#                              spectral_frac_laplacian, gaussian_blur3d
#   B. prior head            : FracPrior (frozen-DoD | learnable-mixture | DoG)
#   C. model zoo             : SegResNetRefine (control), FracPriorNet (arms)
#   D. data pipeline         : decode_*, PatchDataset, sliding-window inference
#   E. metrics               : dice_np, hd95_fast, band_dice, postprocess
#   F. kaggle plumbing       : session budget, run-queue markers, disk guards
# ============================================================================
import glob, hashlib, json, math, os, shutil, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

MODS = ["flair", "t1", "t1ce", "t2"]
DOD_CH = (0, 2)                      # FLAIR, T1ce -- the modalities the DoD arms consume
FULL_VAL_CASES = 37                  # the fixed split has 37 val / 37 test


# ===========================================================================
# A. fractional operators
# ===========================================================================
def _sgn_gamma(x: torch.Tensor) -> torch.Tensor:
    return torch.where(x > 0, torch.ones_like(x), (-1.0) ** (torch.floor(-x) + 1))


def riesz_weights(alpha: torch.Tensor, K: int = 3) -> torch.Tensor:
    """Riesz / fractional centred-difference taps, differentiable in alpha, zero-sum.
    alpha = 2 -> [1,-2,1].  Returns (C, 2K+1)."""
    a = alpha.reshape(-1).to(torch.float64)
    ks = torch.arange(-K, K + 1, dtype=torch.float64, device=a.device)
    x0, xa, xb = a + 1.0, a[:, None] / 2 - ks[None, :] + 1, a[:, None] / 2 + ks[None, :] + 1
    logabs = torch.lgamma(x0)[:, None] - torch.lgamma(xa) - torch.lgamma(xb)
    sgn = _sgn_gamma(x0)[:, None] * _sgn_gamma(xa) * _sgn_gamma(xb) * (-((-1.0) ** ks))
    w = sgn * torch.exp(logabs)
    pole = ((xa <= 0) & (xa == xa.round())) | ((xb <= 0) & (xb == xb.round()))
    w = torch.where(pole, torch.zeros_like(w), w).clone()
    w[:, K] = w[:, K] - w.sum(dim=1)
    return torch.nan_to_num(w, nan=0.0).to(alpha.dtype)


def riesz_weights_np(alpha: float, K: int = 3):
    """numpy twin, for data-side use (matches the reference implementation)."""
    from scipy.special import gamma, rgamma
    ks = np.arange(-K, K + 1)
    g = (-1.0) ** ks * gamma(alpha + 1) * rgamma(alpha / 2 - ks + 1) * rgamma(alpha / 2 + ks + 1)
    w = -g
    w[K] -= w.sum()
    return w, ks


def _conv1d_axis(x, w, K, dim, groups):
    pad = [0] * 6
    pad[2 * (2 - dim)] = K
    pad[2 * (2 - dim) + 1] = K
    xp = F.pad(x, pad, mode="replicate")
    C = x.shape[1]
    shape = [1, 1, 1, 1, 1]
    shape[2 + dim] = 2 * K + 1
    return F.conv3d(xp, w.reshape(shape).expand(C, 1, *shape[2:]).to(x.dtype), groups=C)


class FractionalOrderMixture(nn.Module):
    """Softmax mixture over a fixed grid of orders.  Returns Delta^alpha x with
    E[alpha] = sum_a p_a alpha_a -- the number to report.  Numerically bulletproof
    (no Gamma poles, unlike a continuous alpha)."""

    def __init__(self, alphas=(0.8, 1.0, 1.2, 1.5, 1.8, 2.0), K=3, temperature=1.0,
                 init_probs=None):
        super().__init__()
        self.K = K
        self.register_buffer("alphas", torch.tensor(alphas, dtype=torch.float32))
        self.register_buffer("kernels", torch.stack(
            [torch.tensor(riesz_weights_np(float(a), K)[0], dtype=torch.float32) for a in alphas]))
        self.temperature = temperature
        init = torch.ones(len(alphas)) if init_probs is None else torch.tensor(np.asarray(init_probs), dtype=torch.float32)
        self.logits = nn.Parameter(torch.log(init / init.sum()))

    def posterior(self):
        return torch.softmax(self.logits / self.temperature, dim=0)

    def mean_alpha(self):
        return (self.posterior() * self.alphas).sum()

    def forward(self, x):
        w = self.posterior() @ self.kernels
        return sum(_conv1d_axis(x, w, self.K, d, x.shape[1]) for d in range(3))

    def extra_repr(self):
        return f"E[alpha]={float(self.mean_alpha()):.3f} p={[round(float(v),3) for v in self.posterior()]}"


class RieszConv3d(nn.Module):
    def __init__(self, channels, K=3, alpha_init=1.5, per_channel=True, lo=0.2, hi=2.0):
        super().__init__()
        self.K, self.channels, self.per_channel, self.lo, self.hi = K, channels, per_channel, lo, hi
        n = channels if per_channel else 1
        u = (alpha_init - lo) / (hi - lo)
        self.alpha_raw = nn.Parameter(torch.full((n,), math.log(u / (1 - u))))

    @property
    def alpha(self):
        a = self.lo + (self.hi - self.lo) * torch.sigmoid(self.alpha_raw)
        return a if self.per_channel else a.expand(self.channels)

    def forward(self, x):
        w = riesz_weights(self.alpha, self.K)[:, None, :].to(x.dtype)     # (C,1,2K+1)
        return sum(_conv1d_axis(x, w, self.K, d, x.shape[1]) for d in range(3))


def _frac_symbol(shape, alpha, radial=True, device="cpu"):
    fs = [torch.fft.fftfreq(n, device=device, dtype=torch.float32) * 2 * math.pi for n in shape]
    g = torch.meshgrid(*fs, indexing="ij")
    if radial:
        return -((sum(4.0 * torch.sin(x / 2.0) ** 2 for x in g)) + 1e-12) ** (alpha / 2.0)
    return -sum((4.0 * torch.sin(x / 2.0) ** 2 + 1e-12) ** (alpha / 2.0) for x in g)


def spectral_frac_laplacian(x, alpha, radial=True):
    S = _frac_symbol(x.shape[-3:], alpha, radial, x.device)
    X = torch.fft.fftn(x.float(), dim=(-3, -2, -1))
    return torch.fft.ifftn(X * S, dim=(-3, -2, -1)).real.to(x.dtype)


def gaussian_blur3d(x, sigma, truncate=3.0):
    """separable Gaussian blur, groups=C (the classical anchor for the DoG arm)"""
    if sigma <= 0:
        return x
    r = int(truncate * sigma + 0.5)
    k = torch.exp(-0.5 * (torch.arange(-r, r + 1, device=x.device, dtype=torch.float32) / sigma) ** 2)
    k = (k / k.sum()).to(x.dtype)
    C = x.shape[1]
    out = x
    for d in range(3):
        pad = [0] * 6
        pad[2 * (2 - d)] = r
        pad[2 * (2 - d) + 1] = r
        shape = [1, 1, 1, 1, 1]
        shape[2 + d] = 2 * r + 1
        out = F.conv3d(F.pad(out, pad, mode="replicate"),
                       k.reshape(shape).expand(C, 1, *shape[2:]), groups=C)
    return out


def spectral_dod_volume(x, alpha, T1=4.0, T2=12.0, radial=True, pad=24,
                        signed=False, smooth_sigma=0.0):
    """Exact DoD of one volume, x = (1,1,D,H,W) or (D,H,W) float tensor.
    smooth_sigma > 0 pre-smooths (stand-in for the papers' RBF interpolant).
    Returns |DoD|, or the tuple (|DoD|, signed DoD) when signed=True."""
    u = x.float()[None, None] if x.dim() == 3 else x.float()
    if smooth_sigma and smooth_sigma > 0:
        from scipy.ndimage import gaussian_filter
        u = torch.from_numpy(gaussian_filter(u[0, 0].numpy(), smooth_sigma, mode="nearest"))[None, None]
    up = F.pad(u, [pad] * 6, mode="replicate") if pad else u
    S = _frac_symbol(up.shape[-3:], alpha, radial, "cpu")
    H = (torch.exp(T2 * S) - torch.exp(T1 * S)).float()
    y = torch.fft.ifftn(torch.fft.fftn(up) * H, dim=(-3, -2, -1)).real
    if pad:
        y = y[..., pad:-pad, pad:-pad, pad:-pad]
    return (y.abs(), y) if signed else y.abs()


def match_band_T(alpha, ref_alpha=2.0, ref_T1=4.0, ratio=3.0, radial=True):
    """T1 such that the DoD band centre of `alpha` coincides with the reference
    (alpha=2, T1=4) band centre.  alpha=1.5 -> (2.435, 7.305).  radius=3."""
    k = 2.0 * math.pi / 15.0                      # band centre used by the reference
    ref_sigma = (2.0 * k ** 2) ** (ref_alpha / 2.0) / 2.0 if radial else k ** ref_alpha / 2.0
    sigma = (math.log(ratio) / (ref_T1 * (ratio - 1.0))) if ref_T1 else None
    # numerical solve is more robust than the closed form above
    def peak_w(a, T1):
        ww = np.linspace(1e-4, np.pi, 4000)
        S = -(ww ** a) if not radial else -((2 - 2 * np.cos(ww)) ** (a / 2.0))
        R = np.abs(np.exp(3 * T1 * S) - np.exp(T1 * S))
        return ww[int(np.argmax(R))]
    target = peak_w(ref_alpha, ref_T1)
    lo, hi = 1e-3, 60.0
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if peak_w(alpha, mid) > target:
            lo = mid
        else:
            hi = mid
    T1 = 0.5 * (lo + hi)
    return float(T1), float(T1 * ratio)


# ===========================================================================
# B. prior head -- the operator's only route into the network
# ===========================================================================
class FracPrior(nn.Module):
    """logits_out = logits_in + sum_m beta[c,m] * R(x)[:, m]

    mode="frozen": R = identity on the precomputed DoD channels (x[:, 4:])
    mode="mix"   : R = Delta^{alpha} with a Learnable Mixture over orders, on x[:, :4]
    mode="dog"   : R = x - G_sigma*x  (classical high-pass anchor), on x[:, :4]

    beta is zero-initialised => at step 0 the arm is BIT-IDENTICAL to the control.
    """

    def __init__(self, n_classes=3, n_ch=4, mode="mix", K=3, alpha_init=1.5,
                 dog_sigma=2.0, alphas=(0.8, 1.0, 1.2, 1.5, 1.8, 2.0), init_at_alpha2=True):
        super().__init__()
        self.mode, self.n_ch, self.K = mode, n_ch, K
        self.dog_sigma = dog_sigma
        if mode == "mix":
            init = np.ones(len(alphas))
            if init_at_alpha2:
                init[np.argmin([abs(a - 2.0) for a in alphas])] = 4.0   # start near classical
            self.op = FractionalOrderMixture(alphas=alphas, K=K, init_probs=init)
        else:
            self.op = None
        self.beta = nn.Parameter(torch.zeros(n_classes, n_ch))

    def alpha_value(self):
        return float(self.op.mean_alpha()) if self.op is not None else float("nan")

    def operator(self, x):
        mri = x[:, :4]
        if self.mode == "mix":
            return self.op(mri)
        if self.mode == "dog":
            return mri - gaussian_blur3d(mri, self.dog_sigma)
        return x[:, 4:4 + self.n_ch]                     # frozen precomputed DoD

    def forward(self, logits, x):
        r = self.operator(x)
        s = r.flatten(2).std(dim=2).clamp_min(1e-6).view(r.shape[0], -1, 1, 1, 1)
        r = torch.clamp(r / s, -6.0, 6.0)
        return logits + torch.einsum("cm,bm...->bc...", self.beta, r)

    def extra_repr(self):
        return f"mode={self.mode} n_ch={self.n_ch} " + \
               (f"E[alpha]={self.alpha_value():.3f}" if self.op is not None else "")


# ===========================================================================
# C. model zoo
# ===========================================================================
try:                                                  # noqa: E402
    from monai.networks.nets import SegResNet
    from monai.networks.blocks import Convolution
    from monai.losses import DiceLoss
except ImportError as _e:                            # pragma: no cover
    raise ImportError(
        "MONAI is not installed in this image. Fix it in one of two ways, then re-run from the "
        "top: (a) notebook Settings -> Internet = On and run  !pip install monai  in a cell above; "
        "or (b) use the notebook's Dependency Manager to add 'monai'. Kaggle's standard image has "
        "monai, torch, nibabel, scipy, pandas and scikit-learn pre-installed."
    ) from _e


def _segresnet():
    return SegResNet(spatial_dims=3, in_channels=4, out_channels=3, init_filters=16,
                     blocks_down=(1, 2, 2, 4), blocks_up=(1, 1, 1), dropout_prob=0.2)


class RefineHead(nn.Module):
    """capacity-matched refinement head: sees [coarse probs, input channels]"""

    def __init__(self, in_ch):
        super().__init__()
        last = nn.Conv3d(16, 3, 1)
        nn.init.zeros_(last.weight); nn.init.zeros_(last.bias)
        self.net = nn.Sequential(
            Convolution(spatial_dims=3, in_channels=3 + in_ch, out_channels=16, kernel_size=3,
                        strides=1, norm="instance", act="relu"),
            Convolution(spatial_dims=3, in_channels=16, out_channels=16, kernel_size=3,
                        strides=1, norm="instance", act="relu"),
            last)

    def forward(self, coarse, x):
        return self.net(torch.cat([torch.sigmoid(coarse), x], dim=1))


class FracPriorNet(nn.Module):
    """backbone (MRI only) + refine head + optional operator prior.
    mode=None reproduces the capacity-matched control exactly."""

    def __init__(self, in_ch=4, prior_mode=None, alpha_init=1.5, n_prior_ch=4, dog_sigma=2.0):
        super().__init__()
        self.in_ch = in_ch
        self.backbone = _segresnet()
        self.head = RefineHead(in_ch)
        self.prior = None if prior_mode is None else FracPrior(
            n_classes=3, n_ch=n_prior_ch, mode=prior_mode, alpha_init=alpha_init, dog_sigma=dog_sigma)

    def forward(self, x):
        coarse = self.backbone(x[:, :4])
        logits = coarse + self.head(coarse, x)
        if self.prior is not None:
            logits = self.prior(logits, x)
        return logits

    def alpha_value(self):
        return float("nan") if self.prior is None else self.prior.alpha_value()


def build_model(arm: str, alpha=1.5):
    """arm names used across the notebooks"""
    if arm == "mri":
        return _segresnet()                                     # baseline A
    if arm == "mri_ref":
        return FracPriorNet(in_ch=4, prior_mode=None)            # control B (capacity matched)
    if arm == "mri_ref_dod":
        # CONTROL for the extra channels: same 6-ch input, same head, prior switched OFF.
        # dod15 vs mri_ref_dod therefore differs by beta*R ONLY -> bit-identical at init.
        return FracPriorNet(in_ch=4 + len(DOD_CH), prior_mode=None)
    if arm in ("dod15", "dod20", "dod_shuf"):
        return FracPriorNet(in_ch=4 + len(DOD_CH), prior_mode="frozen", n_prior_ch=len(DOD_CH))
    if arm == "dod_fit":
        return FracPriorNet(in_ch=4, prior_mode="mix")
    if arm == "dod_dog":
        return FracPriorNet(in_ch=4, prior_mode="dog")
    raise ValueError(arm)


ALL_ARMS = ("mri", "mri_ref", "mri_ref_dod", "dod15", "dod20", "dod_fit", "dod_dog", "dod_shuf")
DOD_ARMS = ("mri_ref_dod", "dod15", "dod20", "dod_shuf")


def n_input_channels(arm):
    return 4 + (len(DOD_CH) if arm in DOD_ARMS else 0)


# ===========================================================================
# D. data pipeline
# ===========================================================================
def load_nii(path):
    import nibabel as nib
    return np.nan_to_num(nib.load(path).get_fdata(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)


def list_candidates(pdir, pid, kind, deep=True):
    """Every plausible path for one volume of one case, BEST GUESS FIRST.

    Listing-based and case-INsensitive on purpose.  Real mirrors rename things in ways that
    glob patterns miss: `..._Seg.nii` (capital S), `..._SEG.nii`, `segmentation.nii`,
    `..._seg.nii/..._seg.nii` (the "file" is a folder), a stray suffix, or the bare `seg.nii`.
    A naive `glob(pdir + "*seg*")` returns nothing for the first two, which is what turned a
    single odd case into a dead worker pool (the exception surfaces as IndexError, not as a
    missing-file message).
    """
    exact, base, kind_pid, kind_only = [], [], [], []
    kl, pl = kind.lower(), pid.lower()
    for e in ("nii.gz", "nii", "nii.z", "nii.gz.z", ""):
        nm = f"{pid}_{kind}" + (f".{e}" if e else "")
        for variant in {nm, nm.lower(), nm.upper()}:
            exact.append(os.path.join(pdir, variant))

    def scan(d):
        try:
            entries = sorted(os.listdir(d))
        except OSError:
            return
        for nm in entries:
            full = os.path.join(d, nm)
            if os.path.isdir(full):
                if deep:
                    scan(full)                       # the "file" may be a folder holding it
                continue
            low = nm.lower()
            if low.startswith(pl) and low.startswith(f"{pl}_{kl}"):
                base.append(full)
            elif pl in low and kl in low:
                kind_pid.append(full)
            elif kl in low:
                kind_only.append(full)

    scan(pdir)
    out = []
    for group in (exact, base, kind_pid, kind_only):
        for q in group:
            if os.path.isfile(q) and q not in out:
                out.append(q)
    return out


def case_file(pdir, pid, kind, deep=True):
    """The path to hand to nibabel, or None.  nib.load sniffs the content, so a gzip file
    named `.nii` still loads."""
    cands = list_candidates(pdir, pid, kind, deep=deep)
    return cands[0] if cands else None


def describe_case(pdir, limit=12, kind=None):
    """One-line diagnostic for error messages: what is actually in the case folder (and, for
    the requested volume, the near-misses that were rejected)."""
    try:
        entries = sorted(os.listdir(pdir))
    except OSError as e:
        return f"(cannot list {pdir}: {e})"
    head = entries[:limit]
    more = "" if len(entries) <= limit else f" … +{len(entries) - limit} more"
    txt = f"{pdir} contains {len(entries)} entries: {head}{more}"
    if kind:
        near = [q for q in list_candidates(pdir, pid_of_dir(pdir), kind)
                if os.path.basename(q).lower().find(kind.lower()) >= 0]
        txt += f"\n  names containing {kind!r}: {[os.path.basename(q) for q in near] or 'NONE'}"
        sub = [q for q in sorted(glob.glob(os.path.join(pdir, "**", "*"), recursive=True))
               if os.path.isdir(q)][:5]
        if sub:
            txt += f"\n  subfolders: {[os.path.relpath(q, pdir) for q in sub]}"
    return txt


def pid_of_dir(pdir):
    b = os.path.basename(pdir.rstrip("/"))
    return b if b.startswith("BraTS") else "BraTS20_Training_000"


def find_file(pdir, pid, kind):
    p = case_file(pdir, pid, kind)
    if p is None:
        raise FileNotFoundError(
            f"no '{kind}' volume for {pid}\n  {describe_case(pdir, kind=kind)}\n"
            f"  Accepted: '{pid}_{kind}.nii(.gz)' in any case or spelling, optionally inside a "
            f"folder of the same name, or any file whose name contains '{kind}'.")
    return p


def find_seg(pdir, pid):
    return find_file(pdir, pid, "seg")


def load_case_volumes(pdir, pid, mods=None):
    """(mri_stack, seg) for one case, resolved against whatever the mirror named the files.

    This is what the prep notebook calls: it never constructs a filename by hand, so a
    `.nii`-only mirror, a wrapped-in-folder mirror or an uppercase mirror all work.
    """
    mods = list(mods or MODS)
    vols = [load_nii(find_file(pdir, pid, m)) for m in mods]
    seg = np.rint(load_nii(find_file(pdir, pid, "seg"))).astype(np.uint8)
    seg[seg == 3] = 1                                  # BraTS label convention
    return np.stack(vols), seg


def normalize_mri_single(vol):
    """z-score inside the non-zero mask, clip to the 0.5-99.5 percentile first."""
    m = np.abs(vol) > 1e-6
    out = np.zeros(vol.shape, np.float32)
    if m.sum() < 10:
        return out
    lo, hi = np.percentile(vol[m], [0.5, 99.5])
    v = np.clip(vol[m], lo, hi)
    out[m] = (v - v.mean()) / (v.std() + 1e-8)
    return out


def as_roots(root):
    """A cache may be one dir, a LIST of dirs (Kaggle shards), or a glob pattern."""
    if root is None:
        return []
    items = list(root) if isinstance(root, (list, tuple, set)) else [root]
    out = []
    for r in items:
        cands = sorted(glob.glob(str(r))) if any(c in str(r) for c in "*?[") else [str(r)]
        for c in cands:
            if os.path.isdir(c) and c not in out:
                out.append(c)
    return out


def _first_file(roots, name):
    """the first shard that actually contains `name` (loaders never listdir)."""
    for r in roots:
        f = os.path.join(r, name)
        if os.path.exists(f):
            return f
    return os.path.join(roots[0], name) if roots else name


def cache_file(cache, pid, suffix, ext="npy"):
    return _first_file(as_roots(cache), f"{pid}_{suffix}.{ext}")


def has_case(cache, pid):
    return os.path.exists(cache_file(cache, pid, "mri"))


def loader_case(cache, pid, dod_tag=None, dod_root=None):
    """-> (x float32 (C,D,H,W), seg uint8, dist float32).  Every channel is unit scale."""
    c = as_roots(cache)
    dr = as_roots(dod_root) or c
    x = np.load(_first_file(c, f"{pid}_mri.npy")).astype(np.float32)
    if dod_tag:
        d = np.load(_first_file(dr, f"{pid}_{dod_tag}.npy")).astype(np.float32)
        x = np.concatenate([x, d], axis=0)
    return x, np.load(_first_file(c, f"{pid}_seg.npy")), \
        decode_dist(np.load(_first_file(c, f"{pid}_dist.npy")))


def dod_tag(alpha):
    """canonical file tag for a precomputed DoD: 1.5 -> 'dod_a15', 2.0 -> 'dod_a20'"""
    return f"dod_a{int(round(float(alpha) * 10)):02d}"


def decode_dod(u16, range_=32.0):
    return (u16.astype(np.float32) / 65535.0) * range_


def bbox_crop_slices(mri, margin=4):
    brain = (np.abs(mri) > 1e-6).any(axis=0)
    idx = np.where(brain)
    if len(idx[0]) == 0:
        return tuple(slice(0, s) for s in brain.shape)
    return tuple(slice(max(0, int(idx[a].min()) - margin),
                       min(brain.shape[a], int(idx[a].max()) + 1 + margin)) for a in range(3))


def normalize_mri(mri):
    out = np.zeros(mri.shape, dtype=np.float32)
    for c in range(mri.shape[0]):
        m = np.abs(mri[c]) > 1e-6
        if m.sum() < 10:
            continue
        lo, hi = np.percentile(mri[c][m], [0.5, 99.5])
        v = np.clip(mri[c][m], lo, hi)
        out[c][m] = (v - v.mean()) / (v.std() + 1e-8)
    return out


def seg_to_targets(seg):
    return np.stack([seg > 0, (seg == 1) | (seg == 4), seg == 4]).astype(np.float32)


def signed_distance_to_wt(seg, clip=40.0):
    from scipy.ndimage import distance_transform_edt
    wt = seg > 0
    if not wt.any():
        return np.zeros(seg.shape, np.float32)
    return np.clip(distance_transform_edt(~wt) - distance_transform_edt(wt), -clip, clip).astype(np.float32)


def encode_dist(d, clip=40.0):
    """signed distance -> uint8 (4.8 MB/case instead of 9.6).  80 voxels / 255 levels
    = 0.31 voxel resolution, which is far finer than the tau=3 weighting needs."""
    z = np.clip((np.asarray(d, np.float32) + clip) / (2 * clip), 0, 1)
    return np.rint(z * 255).astype(np.uint8)


def decode_dist(u, clip=40.0):
    return (np.asarray(u, np.float32) / 255.0) * (2 * clip) - clip


def sample_locs(seg, n=3000, seed=0):
    rng = np.random.default_rng(seed)
    d = {}
    for name, cond in (("c1", seg == 1), ("c2", seg == 2), ("c4", seg == 4)):
        co = np.argwhere(cond)
        if len(co) > n:
            co = co[rng.choice(len(co), n, replace=False)]
        d[name] = co.astype(np.int16)
    return d


class PatchDataset(torch.utils.data.Dataset):
    """Random patch sampler.  Sample i is a pure function of (SEED, offset+i) so every
    arm sees the identical sample sequence -> the comparison is paired by construction.

    Returns (x[C,D,H,W], y[3,D,H,W], dist[1,D,H,W]).  x channels are ALL unit scale.
    """

    P_FG = 0.70

    def __init__(self, cache, ids, arm, length, offset=0, seed=42, patch=(96, 96, 96),
                 dod_tag=None, dod_root=None, shuffle_dod=False, jitter=True):
        self.cache, self.ids, self.arm = as_roots(cache), list(ids), arm
        self.length, self.offset, self.seed, self.patch = length, offset, seed, tuple(patch)
        # dod_tag: which precomputed DoD to load, e.g. "dod_a15" / "dod_a20". None = no DoD.
        # dod_root: the DoD is a SEPARATE dataset from the MRI cache on Kaggle.
        self.dod_tag, self.shuffle_dod, self.jitter = dod_tag, shuffle_dod, jitter
        self.dod_root = as_roots(dod_root) or self.cache
        self.use_dod = dod_tag is not None

    def __len__(self):
        return self.length

    def _p(self, pid, s, ext="npy"):
        return _first_file(self.cache, f"{pid}_{s}.{ext}")

    def __getitem__(self, i):
        rng = np.random.default_rng([self.seed, self.offset + i])
        pid = self.ids[int(rng.integers(len(self.ids)))]
        mri = np.load(self._p(pid, "mri"), mmap_mode="r")
        seg = np.load(self._p(pid, "seg"), mmap_mode="r")
        dist = np.load(self._p(pid, "dist"), mmap_mode="r")
        shape = seg.shape

        center = None
        if rng.random() < self.P_FG:
            locs = np.load(self._p(pid, "locs", "npz"))
            present = [k for k in ("c1", "c2", "c4") if len(locs[k]) > 0]
            if present:
                k = present[int(rng.integers(len(present)))]
                center = locs[k][int(rng.integers(len(locs[k])))]
        if center is None:
            center = [rng.integers(s) for s in shape]
        st = [int(np.clip(c - p // 2, 0, max(0, s - p))) for c, s, p in zip(center, shape, self.patch)]
        sl = tuple(slice(a, a + p) for a, p in zip(st, self.patch))

        x = np.asarray(mri[(slice(None),) + sl], dtype=np.float32)
        if self.use_dod:
            src = pid
            if self.shuffle_dod:                      # placebo arm: DoD from ANOTHER case
                src = self.ids[int(rng.integers(len(self.ids)))]
            d = np.load(_first_file(self.dod_root, f"{src}_{self.dod_tag}.npy"), mmap_mode="r")
            d = np.stack([np.asarray(d[c][sl], dtype=np.float32) for c in range(d.shape[0])])
            p = [(0, 0)] + [(0, q - t) for q, t in zip(self.patch, d.shape[1:])]
            if any(b for _, b in p[1:]):
                d = np.pad(d, p)
            x = np.concatenate([x, d], axis=0)
        y = seg_to_targets(np.asarray(seg[sl]))
        dd = decode_dist(np.asarray(dist[sl]))[None]

        pad = [(0, 0)] + [(0, p - s) for p, s in zip(self.patch, x.shape[1:])]
        if any(b for _, b in pad):
            x = np.pad(x, pad)
            y = np.pad(y, [(0, 0)] + pad[1:])
            dd = np.pad(dd, [(0, 0)] + pad[1:])
        for ax in (1, 2, 3):
            if rng.random() < 0.5:
                x, y, dd = np.flip(x, ax), np.flip(y, ax), np.flip(dd, ax)
        if self.jitter:
            for c in range(min(4, x.shape[0])):       # MRI channels only
                if rng.random() < 0.5:
                    m = x[c] != 0
                    x[c] = np.where(m, x[c] * rng.uniform(0.9, 1.1) + rng.uniform(-0.1, 0.1), 0.0)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y)),
                torch.from_numpy(np.ascontiguousarray(dd)))


# ===========================================================================
# E. losses / metrics / inference
# ===========================================================================
def distance_weighted_bce(logits, y, dist, tau=3.0, peak=5.0):
    w = 1.0 + peak * torch.exp(-dist.abs() / tau)
    return F.binary_cross_entropy_with_logits(logits, y, weight=w.expand_as(y), reduction="mean")


def _perimeter_weights(shape, alpha, device):
    fs = [torch.fft.fftfreq(shape[0], device=device, dtype=torch.float32) * 2 * math.pi,
          torch.fft.fftfreq(shape[1], device=device, dtype=torch.float32) * 2 * math.pi,
          torch.fft.rfftfreq(shape[2], device=device, dtype=torch.float32) * 2 * math.pi]
    g = torch.meshgrid(*fs, indexing="ij")
    k = torch.sqrt(sum(x ** 2 for x in g))
    w = (k / (k.max() + 1e-9)) ** alpha
    return w / (w.mean() + 1e-12)


class FractPerimLoss(nn.Module):
    """|k|^alpha seminorm of the probability map, normalised by the target's own
    seminorm -> one lambda is comparable across alpha."""

    def __init__(self, alpha=1.5):
        super().__init__()
        self.alpha = alpha

    def _semi(self, p):
        w = _perimeter_weights(p.shape[-3:], self.alpha, p.device)
        q = p.float() - p.float().mean(dim=(-3, -2, -1), keepdim=True)
        P = torch.fft.rfftn(q, dim=(-3, -2, -1)).abs() ** 2
        return (P * w).sum(dim=(-3, -2, -1)).mean() / float(np.prod(p.shape[-3:]))

    def forward(self, prob, target):
        with torch.no_grad():
            L0 = self._semi(target)
        return self._semi(prob) / (L0 + 1e-8)


class SegLoss(nn.Module):
    def __init__(self, boundary=True, frac_alpha=None, lam_frac=0.0):
        super().__init__()
        self.dice = DiceLoss(sigmoid=True, squared_pred=True, smooth_nr=0.0, smooth_dr=1e-5, batch=True)
        self.boundary = boundary
        self.frac = FractPerimLoss(frac_alpha) if (frac_alpha and lam_frac > 0) else None
        self.lam_frac = lam_frac

    def forward(self, logits, y, dist=None):
        loss = self.dice(logits, y)
        loss = loss + (distance_weighted_bce(logits, y, dist) if (dist is not None and self.boundary)
                       else F.binary_cross_entropy_with_logits(logits, y))
        if self.frac is not None:
            loss = loss + self.lam_frac * self.frac(torch.sigmoid(logits), y)
        return loss


def dice_np(p, t):
    ps, ts = int(p.sum()), int(t.sum())
    return 1.0 if ps + ts == 0 else 2.0 * int((p & t).sum()) / (ps + ts)


def hd95(pred, gt, penalty=373.13):
    from scipy.ndimage import binary_erosion, distance_transform_edt
    ps, gs = bool(pred.any()), bool(gt.any())
    if not ps and not gs:
        return 0.0
    if ps != gs:
        return penalty
    idx = np.where(pred | gt)
    sl = tuple(slice(max(0, int(i.min()) - 1), int(i.max()) + 2) for i in idx)
    p, g = pred[sl], gt[sl]
    sp, sg = p & ~binary_erosion(p), g & ~binary_erosion(g)
    d = np.concatenate([distance_transform_edt(~g)[sp], distance_transform_edt(~p)[sg]])
    return float(np.percentile(d, 95))


def band_mask(mask, r=3):
    from scipy.ndimage import binary_dilation, binary_erosion
    if not mask.any():
        return np.zeros_like(mask)
    return binary_dilation(mask, iterations=r) & ~binary_erosion(mask, iterations=r)


def band_dice(pred, gt, r=3):
    b = band_mask(gt, r)
    return float("nan") if not b.any() else dice_np(pred & b, gt & b)


def clean_components(mask, min_frac=0.1):
    from scipy.ndimage import label
    if not mask.any() or min_frac <= 0:
        return mask
    lab, n = label(mask)
    if n <= 1:
        return mask
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    keep = sizes >= min_frac * sizes.max(); keep[0] = False
    return keep[lab]


def postprocess(pred, thr=None, cc_wt=0.0, cc_tc=0.0, et_min=0):
    """THE single place where post-processing happens (val, test and tuning all call this)."""
    out = np.zeros_like(pred, bool)
    for c, cc in ((0, cc_wt), (1, cc_tc)):
        m = pred[c] if thr is None else pred[c]
        out[c] = clean_components(m, cc) if cc > 0 else m
    e = pred[2]
    out[2] = e if int(e.sum()) >= et_min else np.zeros_like(e)
    return out


@torch.no_grad()
def infer_probs(model, x_np, patch, overlap, device, amp=True, sw_batch=2):
    try:
        from monai.inferers import sliding_window_inference
    except ImportError as _e:                        # pragma: no cover
        raise ImportError("MONAI (monai.inferers) is missing -> !pip install monai, then re-run.") from _e
    model.eval()
    xt = torch.from_numpy(x_np)[None].to(device)
    with torch.amp.autocast("cuda", enabled=amp, dtype=torch.float16):
        o = sliding_window_inference(xt, patch, sw_batch_size=sw_batch, predictor=model,
                                     overlap=overlap, mode="gaussian")
    return torch.sigmoid(o.float())[0].cpu().numpy().astype(np.float16)


# ===========================================================================
# F. kaggle plumbing
# ===========================================================================
class SessionBudget:
    """Graceful wall-clock guard: the notebook always exits cleanly and saves its
    checkpoints before the 9 h GPU cap hits."""

    def __init__(self, hours):
        self.t0 = time.time()
        self.limit = hours * 3600

    def left(self):
        return self.limit - (time.time() - self.t0)

    def ok(self, need=0.0):
        return self.left() > need

    def hhmm(self):
        s = max(0, int(self.left()))
        return f"{s//3600}h{(s%3600)//60:02d}m"


def run_done(run_dir):
    return os.path.exists(os.path.join(run_dir, "DONE"))


def mark_done(run_dir, payload=None):
    os.makedirs(run_dir, exist_ok=True)
    if payload is not None:
        json.dump(payload, open(os.path.join(run_dir, "DONE"), "w"), indent=2)
    else:
        open(os.path.join(run_dir, "DONE"), "w").write("ok")


def adopt_previous_runs(src, dst):
    """Copy everything from an attached dataset (previous sessions' output) into the
    working dir so the run queue can skip finished runs.  This is what makes the
    study resumable across Kaggle sessions, which do not persist /kaggle/working."""
    if not src or not os.path.isdir(src):
        print(f"adopt: nothing at {src}")
        return
    n = 0
    for root, _, files in os.walk(src):
        rel = os.path.relpath(root, src)
        for f in files:
            d = os.path.join(dst, rel, f)
            if not os.path.exists(d):
                os.makedirs(os.path.dirname(d), exist_ok=True)
                shutil.copy2(os.path.join(root, f), d)
                n += 1
    print(f"adopt: {n} new file(s) copied from {src} -> {dst}")


def disk_report(*paths):
    for p in paths:
        if os.path.isdir(p):
            tot = sum(os.path.getsize(os.path.join(r, f))
                      for r, _, fs in os.walk(p) for f in fs) / 1e9
            print(f"  {p:55s} {tot:6.2f} GB")
        elif os.path.exists(p):
            print(f"  {p:55s} {os.path.getsize(p)/1e6:6.1f} MB")
        else:
            print(f"  {p:55s} (missing)")


def peak_gpu_mem_gb():
    return torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0

"""safepp.py -- "safe post-processing": the mathematical core of the glioma reading aid.

Standalone (numpy + scipy only, no torch, no GPU, no Kaggle).  Everything here is
deterministic, unit-testable, and calibrated on VALIDATION data only.

Contents
--------
A. morphology / post-processing operators
     components()               26-connected component labelling
     filter_components()        remove components below a physical size
     relabel_et_to_necrosis()   nnU-Net BraTS post-processing step
     postproc_seg()             the two operators as one validated pipeline

B. risk control  (Angelopoulos et al., Conformal Risk Control, ICLR 2024)
     miss_curve()               per-case "fraction of tumour missed" vs threshold
     crc_threshold()            lambda-hat with the finite-sample correction
     stratified_crc()           group-conditional calibration per stratum

C. certified component filtering
     removal_curve()            fraction of true lesion volume a filter would delete
     certified_filter_threshold()

D. metrics
     dice_regions(), hd95(), lesionwise_dice(), component_stats(), volume_error()

Every public function has a docstring stating what it guarantees and what it does NOT
guarantee -- read those before quoting a number in a presentation.

Run `python3 safepp.py` for the self-test suite (no arguments, no data needed).
"""

import numpy as np
from scipy import ndimage

# --------------------------------------------------------------------------------------
# A. post-processing operators
# --------------------------------------------------------------------------------------

CONN26 = np.ones((3, 3, 3), dtype=bool)          # 26-connectivity for 3D components

# BraTS label convention used throughout the harness
NCR, ED, ET = 1, 2, 4
REGIONS = {"WT": (1, 2, 4), "TC": (1, 4), "ET": (4,)}


def labels_from_probs(p_wt, p_tc, p_et, thr):
    """Nested BraTS labels from three foreground probability maps.

    One threshold, three maps, nested by construction: ET sits inside TC sits inside WT, so
    the label map can never contain an ET island outside the tumour.  This is the ONLY place
    where probabilities become a mask -- val, calibration and test all call it, which is what
    makes the certified threshold meaningful.
    """
    wt, tc, et = p_wt >= thr, p_tc >= thr, p_et >= thr
    seg = np.zeros(p_wt.shape, np.uint8)
    seg[wt] = ED
    seg[wt & tc] = NCR
    seg[wt & tc & et] = ET
    return seg


def components(mask, conn=CONN26):
    """Label 26-connected components of a binary mask.

    Returns (labels int32, n_components).  `labels == 0` is background.
    """
    lab, n = ndimage.label(np.asarray(mask, dtype=bool), structure=conn)
    return lab.astype(np.int32), int(n)


def component_sizes(mask, conn=CONN26):
    """Sorted-descending array of component sizes (voxels).  Empty if mask is empty."""
    lab, n = components(mask, conn)
    if n == 0:
        return np.zeros(0, dtype=np.int64)
    return np.bincount(lab.ravel())[1:].astype(np.int64)


def filter_components(mask, min_voxels, conn=CONN26):
    """Keep only components with >= `min_voxels` voxels.

    Deterministic, idempotent, and monotone: raising the threshold can only remove more
    voxels.  That monotonicity is what makes the certified variant in section C valid.
    """
    mask = np.asarray(mask, dtype=bool)
    if min_voxels <= 1:
        return mask.copy()
    lab, n = components(mask, conn)
    if n == 0:
        return mask.copy()
    keep = np.bincount(lab.ravel(), minlength=n + 1) >= min_voxels
    keep[0] = False
    return keep[lab]


def relabel_et_to_necrosis(seg, min_et_voxels, ncr=NCR, et=ET):
    """nnU-Net's BraTS post-processing: if the enhancing tumour is smaller than
    `min_et_voxels`, relabel it to non-enhancing tumour/necrosis.

    Rationale (Isensee et al., BraTS 2020): tiny ET predictions are usually false
    positives near vessels/choroid plexus, and the challenge metric scores an empty
    prediction in an ET-less case as perfect.  Their paper also warns this *removes*
    true small enhancements, which is exactly why we certify it in section C.
    """
    seg = np.asarray(seg).copy()
    if min_et_voxels is None or min_et_voxels <= 0:
        return seg
    et_mask = seg == et
    if int(et_mask.sum()) < min_et_voxels:
        seg[et_mask] = ncr
    return seg


def postproc_seg(seg, params):
    """Apply the validated post-processing pipeline to a label map.

    params: dict(min_vox={region: int}, min_et_voxels=int, keep_largest=bool)
            `min_vox` is applied per region on the region mask; a voxel is removed only
            if it is not part of *any* kept region, so WT/TC/ET stay nested.
    """
    seg = np.asarray(seg).copy()
    min_vox = (params or {}).get("min_vox") or {}
    # 1. region-wise size filtering, nested: compute keep masks then re-derive labels
    wt_keep = filter_components(np.isin(seg, REGIONS["WT"]), int(min_vox.get("WT", 1)))
    tc_keep = filter_components(np.isin(seg, REGIONS["TC"]), int(min_vox.get("TC", 1)))
    et_keep = filter_components(np.isin(seg, REGIONS["ET"]), int(min_vox.get("ET", 1)))
    if (params or {}).get("keep_largest_wt"):
        lab, n = components(wt_keep)
        if n > 1:
            sizes = np.bincount(lab.ravel(), minlength=n + 1)
            wt_keep = lab == int(np.argmax(sizes[1:]) + 1)
    out = np.zeros_like(seg)
    out[wt_keep] = seg[wt_keep]                     # keep original labels inside WT
    out[~wt_keep] = 0
    et_removed = np.isin(seg, REGIONS["ET"]) & ~et_keep & wt_keep
    out[et_removed] = NCR                           # drop ET but keep it as core
    tc_removed = np.isin(seg, REGIONS["TC"]) & ~tc_keep
    out[tc_removed & (out != 0)] = ED               # drop TC -> falls back to edema
    # 2. ET relabeling last, so it sees the cleaned ET
    out = relabel_et_to_necrosis(out, (params or {}).get("min_et_voxels", 0))
    return out


# --------------------------------------------------------------------------------------
# B. conformal risk control
# --------------------------------------------------------------------------------------

def _as_prob(p):
    p = np.asarray(p, dtype=np.float32)
    return p


def miss_curve(prob, gt_fg, thresholds):
    """Per-threshold "fraction of ground-truth tumour voxels missed".

    prob    : (D,H,W) foreground probability
    gt_fg   : (D,H,W) boolean ground truth
    returns : array of shape (len(thresholds),) -- risk for "predict tumour if p >= t"

    Risk is non-decreasing in the threshold t (a stricter threshold recalls less), and
    bounded in [0, 1].  Empty ground truth -> risk 0 (nothing can be missed).
    """
    prob, gt_fg = _as_prob(prob), np.asarray(gt_fg, dtype=bool)
    n_fg = int(gt_fg.sum())
    if n_fg == 0:
        return np.zeros(len(thresholds), dtype=np.float64)
    v = prob[gt_fg]
    return np.array([float((v < t).mean()) for t in thresholds], dtype=np.float64)


def crc_threshold(calib_risks, alpha, B=1.0, thresholds=None):
    """Conformal risk control threshold with the finite-sample correction.

    calib_risks : (n_cal, n_thresholds) per-calibration-case risk, non-decreasing in the
                  threshold axis (as produced by `miss_curve`).
    alpha       : target risk level in (0, 1)
    B           : upper bound on the loss (1.0 for a fraction)

    Returns dict(threshold, lam, feasible, mean_risk_cal, bound_cal, risk_curve,
                 thresholds) where the guarantee is

        E[ risk(new case, threshold) ] <= alpha      (expectation over the calibration draw)

    Why the correction: with a finite calibration set the plain empirical quantile is
    optimistic; CRC adds B/(n+1) and rescales by n/(n+1), which yields the bound above
    with no distributional assumptions (Angelopoulos et al. 2024, Theorem 1).

    What it does NOT give: per-case coverage, or coverage inside a subgroup -- that is
    `stratified_crc`.  If no threshold satisfies the bound the function returns the most
    conservative threshold it has (t = min grid, i.e. the largest prediction set) with
    feasible=False: report that the model cannot meet this risk level, do not pretend.
    """
    R = np.asarray(calib_risks, dtype=np.float64)
    n = R.shape[0]
    if n == 0:
        raise ValueError("empty calibration set")
    if thresholds is None:
        thresholds = np.linspace(0, 1, R.shape[1])
    thresholds = np.asarray(thresholds, dtype=np.float64)
    assert R.shape[1] == len(thresholds), \
        f"risk array has {R.shape[1]} columns but {len(thresholds)} thresholds were given"
    mean_risk = R.mean(axis=0)
    bound = (n / (n + 1.0)) * mean_risk + B / (n + 1.0)
    ok = np.where(bound <= alpha)[0]
    if len(ok):
        # Tightest set that still meets the target: risk is non-decreasing in the
        # threshold, so the LARGEST feasible threshold is the smallest prediction set
        # that keeps the guarantee.  Taking ok[0] instead would return "predict
        # everything", which is a vacuous certificate.
        j = int(ok[-1])
        feasible = True
    else:
        j, feasible = 0, False            # no threshold meets alpha -> say so, don't fake it
    return dict(threshold=float(thresholds[j]), lam=float(1.0 - thresholds[j]),
                feasible=feasible, mean_risk_cal=float(mean_risk[j]),
                bound_cal=float(bound[j]), risk_curve=mean_risk, bound_curve=bound,
                thresholds=thresholds, n_cal=n)


def stratified_crc(calib_risks_by_group, alpha, B=1.0, thresholds=None):
    """Group-conditional CRC: one threshold per stratum.

    Group-conditional calibration gives E[risk | group] <= alpha for each group, instead
    of only E[risk] <= alpha overall.  The statistical price is a smaller calibration set
    per group (so a looser, larger prediction set) -- report that price, it is the honest
    part of the claim.

    returns {group: crc_threshold(...)} and, for convenience, the pooled threshold under
    key "__pooled__".
    """
    out = stratified_crc.__wrapped__ if False else {}
    pooled = np.concatenate([np.asarray(v) for v in calib_risks_by_group.values()], axis=0)
    out["__pooled__"] = crc_threshold(pooled, alpha, B=B, thresholds=thresholds)
    for g, R in calib_risks_by_group.items():
        out[g] = crc_threshold(np.asarray(R), alpha, B=B, thresholds=thresholds)
    return out


def fp_curve(prob, gt_fg, thresholds):
    r"""Fraction of the PREDICTION that is wrong: |pred_theta \ GT| / |pred_theta|.

    This is the over-segmentation counterpart of `miss_curve`: it FALLS as the threshold
    rises (a stricter threshold throws away the least confident -- usually spurious --
    voxels first).  Empty prediction -> 0 by convention (nothing predicted, nothing wrong).
    """
    prob, gt_fg = _as_prob(prob), np.asarray(gt_fg, dtype=bool)
    out = []
    for t in thresholds:
        pred = prob >= t
        n = int(pred.sum())
        out.append(0.0 if n == 0 else float((pred & ~gt_fg).sum()) / n)
    return np.array(out, dtype=np.float64)


def monotone_envelope(r, direction="increasing"):
    """Replace a noisy risk curve by its worst-case monotone envelope.

    Certifying the envelope certifies the deployed operator, because at every threshold
    the envelope is >= the realised risk:
      * increasing family -> running MAXIMUM over smaller thresholds
      * decreasing family -> running MAXIMUM over larger thresholds
    The alternative (`minimum.accumulate`) looks tighter and is WRONG: it would certify a
    risk lower than the operator actually carries.
    """
    r = np.asarray(r, dtype=np.float64)
    # ALONG THE THRESHOLD AXIS ONLY.  Calling np.maximum.accumulate without `axis` on a
    # (n_cal, n_thresholds) matrix flattens it in C order, so the running maximum leaks
    # from one calibration case into the next and the bounds inflate by a large factor
    # (observed: mean risk 0.055 -> envelope 0.34), which makes every certificate
    # infeasible and the deployed threshold needlessly conservative.
    if direction == "increasing":
        return np.maximum.accumulate(r, axis=-1)
    return np.maximum.accumulate(r[..., ::-1], axis=-1)[..., ::-1]


def certified_threshold_pair(calib_fn, calib_fp, thresholds, alpha, beta, B=1.0):
    """Two one-sided certificates -> one admissible threshold band.

    calib_fn : (n_cal, K) per-case MISSED-tumour fraction, increasing in the threshold
    calib_fp : (n_cal, K) per-case WRONG-prediction fraction, decreasing in the threshold
    alpha    : bound on the missed fraction      (under-segmentation risk)
    beta     : bound on the wrong-prediction fraction (over-segmentation risk)

    Rule (pre-registered, decided before looking at test): take the tightest threshold
    that meets the missed-tumour target, and accept it only if it also meets the
    wrong-prediction target.  If the two targets cannot be met together at this calibration
    size, `feasible` is False and you report that instead of a number.
    """
    F = monotone_envelope(np.asarray(calib_fn), "increasing")
    G = monotone_envelope(np.asarray(calib_fp), "decreasing")
    assert F.shape == G.shape, "risk arrays must have the same shape"
    n = F.shape[0]
    thresholds = np.asarray(thresholds, dtype=np.float64)
    assert F.shape[1] == len(thresholds), "risk arrays and thresholds disagree"
    fn_bound = (n / (n + 1.0)) * F.mean(axis=0) + B / (n + 1.0)
    fp_bound = (n / (n + 1.0)) * G.mean(axis=0) + B / (n + 1.0)
    ok_fn = np.where(fn_bound <= alpha)[0]        # prefix: risk grows with the threshold
    ok_fp = np.where(fp_bound <= beta)[0]         # suffix: risk falls with the threshold
    j_fn = int(ok_fn[-1]) if len(ok_fn) else None
    j_fp = int(ok_fp[0]) if len(ok_fp) else None
    feasible = (j_fn is not None) and (j_fp is not None) and (j_fn >= j_fp)
    j = j_fn if j_fn is not None else 0
    return dict(lam=float(thresholds[j]), feasible=bool(feasible),
                thr_fn=float(thresholds[j_fn]) if j_fn is not None else None,
                thr_fp=float(thresholds[j_fp]) if j_fp is not None else None,
                fn_bound_curve=fn_bound, fp_bound_curve=fp_bound,
                fn_mean_curve=F.mean(axis=0), fp_mean_curve=G.mean(axis=0),
                thresholds=thresholds, alpha=float(alpha), beta=float(beta), n_cal=n)


def choose_certificate(cert, cnr=None, stratified=True):
    """Which certificate should be presented for ONE case?

    The pooled certificate is valid on average over the cohort; a stratified one is valid
    within its subgroup and is therefore the right thing to show when it exists, is feasible,
    and we know the case's contrast.  Falling back to the pooled certificate is silently safe:
    the caller never gets a number out of a stratum whose certificate was infeasible.

    cert : the per-arm entry of certificates.json
    cnr  : the case's contrast-to-noise (wt_cnr); None/NaN -> pooled
    Returns dict(scope, lam, n_cal, curves, pooled_lam, cut) with scope 'low'|'high'|'pooled'.
    """
    strata = (cert or {}).get("strata") or None
    pooled = dict(scope="pooled", lam=(cert or {}).get("lam"), n_cal=(cert or {}).get("n_cal"),
                  curves=(cert or {}).get("curves"), pooled_lam=(cert or {}).get("lam"),
                  cut=(strata or {}).get("cut"))
    if not (stratified and strata) or cnr is None or not np.isfinite(float(cnr)):
        return pooled
    low = float(cnr) <= float(strata["cut"])
    lam = strata.get("lam_low" if low else "lam_high")
    feas = bool(strata.get("feasible_low" if low else "feasible_high"))
    if not (feas and lam is not None):
        return pooled                                   # stratum not certifiable -> pooled
    cur = strata.get("curves_low" if low else "curves_high") or pooled["curves"]
    return dict(scope=("low" if low else "high"), lam=float(lam),
                n_cal=int(strata["n_low" if low else "n_high"]), curves=cur,
                pooled_lam=(cert or {}).get("lam"), cut=float(strata["cut"]))


def volume_interval(v_pred, alpha, beta):
    """Worst-case volume interval implied by the two certified risk levels.

    If a case's missed fraction is at most alpha and its wrong-prediction fraction at most
    beta, then
        true >= (1 - beta) * predicted      (at least that share of the prediction is real)
        true <= predicted / (1 - alpha)     (the missed tumour cannot exceed that share)
    so the interval is guaranteed *given* those per-case risk levels.

    IMPORTANT: the certificates bound the AVERAGE risk over cases, not each individual
    case, so this is the worst-case consequence for a case sitting at the certified level.
    Always report the empirical containment rate on held-out cases next to it.
    """
    hi = float(v_pred) / max(1e-9, (1.0 - float(alpha)))
    lo = float(v_pred) * (1.0 - float(beta))
    return (min(lo, hi), max(lo, hi))


def volume_at(prob, gt_fg, threshold, voxel_vol_mm3=1.0):
    """Predicted volume (mm^3) at a threshold, with the ground-truth volume alongside."""
    prob = _as_prob(prob)
    pred = prob >= threshold
    return (float(pred.sum()) * voxel_vol_mm3,
            float(np.asarray(gt_fg, bool).sum()) * voxel_vol_mm3)


def per_case_risk(prob, gt_fg, threshold):
    """Realised risk for one case at one threshold (use this to audit a certificate)."""
    return float(miss_curve(prob, gt_fg, [threshold])[0])


# --------------------------------------------------------------------------------------
# C. certified component filtering
# --------------------------------------------------------------------------------------

def removal_curve(pred_mask, gt_seg, thresholds, region="WT", voxel_vol_mm3=1.0):
    """Per-threshold fraction of ground-truth lesion volume that filtering would delete.

    For each threshold d (in voxels): apply `filter_components` to the predicted region
    mask and measure the ground-truth voxels that the FILTER deleted -- i.e.
    (prediction before AND NOT after) AND ground truth, divided by the ground-truth
    volume.  The result is the filter's collateral damage, per case, in [0, 1].

    Note the subtlety (it changes the clinical claim): voxels the model never predicted
    are NOT charged to the filter, otherwise model error would masquerade as filter risk.

    Non-decreasing in d (see `filter_components` monotonicity), which is what makes the
    CRC bound below valid.
    """
    pred = np.isin(pred_mask, REGIONS[region]) if pred_mask.dtype != bool else pred_mask
    gt = np.isin(gt_seg, REGIONS[region])
    n_gt = int(gt.sum())
    if n_gt == 0:
        return np.zeros(len(thresholds), dtype=np.float64)
    out = []
    for d in thresholds:
        kept = filter_components(pred, int(d))
        removed = pred & ~kept                       # what the filter took away
        out.append(float((removed & gt).sum()) / n_gt)
    return np.array(out, dtype=np.float64)


def certified_filter_threshold(calib_removal, alpha, B=1.0, thresholds=None,
                              component_size_floor=None):
    """Largest component-size threshold whose collateral damage is risk-controlled.

    calib_removal : (n_cal, n_thresholds) from `removal_curve`, non-decreasing in d.
    Guarantee: E[deleted true-lesion volume fraction] <= alpha for a new case.
    component_size_floor : optional cap (voxels) on the returned threshold.  Needed
        because the guarantee is EXPECTATIONAL: if the calibration set happens to
        contain no true lesion between 60 and 1600 voxels, the damage curve is flat
        there and the certified threshold can land just below the first big lesion --
        statistically valid, clinically unnerving.  Set the floor from the observed
        distribution of true component sizes on validation (e.g. its 25th percentile)
        and report both numbers: "certified delta = X, engineering floor = Y".

    Because `filter_components` is monotone, the family of operators is nested, so the
    same CRC correction applies (index the family by -d, or equivalently search for the
    LARGEST d that satisfies the bound -- done below).
    """
    R = np.asarray(calib_removal, dtype=np.float64)
    n = R.shape[0]
    if thresholds is None:
        thresholds = np.arange(R.shape[1])
    thresholds = np.asarray(thresholds, dtype=np.float64)
    assert R.shape[1] == len(thresholds), \
        f"removal array has {R.shape[1]} columns but {len(thresholds)} thresholds were given"
    mean_risk = R.mean(axis=0)
    bound = (n / (n + 1.0)) * mean_risk + B / (n + 1.0)
    ok = np.where(bound <= alpha)[0]
    if len(ok):
        j = int(ok[-1])                   # LARGEST threshold that is still safe
        feasible = True
    else:
        j, feasible = 0, False
    delta, capped = float(thresholds[j]), False
    if component_size_floor is not None and delta > float(component_size_floor):
        delta, capped = float(component_size_floor), True
    return dict(threshold=delta, feasible=feasible, capped=capped,
                certified_threshold=float(thresholds[j]),
                mean_risk_cal=float(mean_risk[j]), bound_cal=float(bound[j]),
                risk_curve=mean_risk, bound_curve=bound, thresholds=thresholds, n_cal=n)


# --------------------------------------------------------------------------------------
# D. metrics
# --------------------------------------------------------------------------------------

def dice(a, b, eps=1e-8):
    a, b = np.asarray(a, dtype=bool), np.asarray(b, dtype=bool)
    return float(2.0 * (a & b).sum() / (a.sum() + b.sum() + eps))


def dice_regions(pred_seg, gt_seg):
    """{WT, TC, ET} voxel-wise Dice.  Empty-prediction convention: if both are empty the
    Dice is defined as 1.0 (the BraTS convention used in the nnU-Net paper)."""
    out = {}
    for r, labs in REGIONS.items():
        p, g = np.isin(pred_seg, labs), np.isin(gt_seg, labs)
        if p.sum() == 0 and g.sum() == 0:
            out[r] = 1.0
        else:
            out[r] = dice(p, g)
    return out


def _surf_dist(a, b, spacing=(1.0, 1.0, 1.0)):
    """Symmetric surface distances: for each surface voxel of a, distance to b (EDT)."""
    if a.sum() == 0 or b.sum() == 0:
        return None
    dt_b = ndimage.distance_transform_edt(~b, sampling=spacing)
    dt_a = ndimage.distance_transform_edt(~a, sampling=spacing)
    sa = a & ~ndimage.binary_erosion(a)          # 6-connectivity surface
    sb = b & ~ndimage.binary_erosion(b)
    if sa.sum() == 0 or sb.sum() == 0:
        return None
    d_ab, d_ba = dt_b[sa], dt_a[sb]
    return np.concatenate([d_ab, d_ba])


def hd95(pred_seg, gt_seg, region="WT", spacing=(1.0, 1.0, 1.0)):
    """95th percentile of the symmetric surface distance, in mm (NaN if undefined).

    Convention: if exactly one side is empty the HD95 is NaN -- do not silently score it
    as 0 or as a huge number; the caller decides (report the count of such cases).
    """
    p, g = np.isin(pred_seg, REGIONS[region]), np.isin(gt_seg, REGIONS[region])
    d = _surf_dist(p, g, spacing)
    return float("nan") if d is None else float(np.percentile(d, 95))


def lesionwise_dice(pred_seg, gt_seg, region="WT", min_gt_voxels=1, bbox_dilate=0):
    """Lesion-wise Dice, averaged over ground-truth connected components of `region`.

    Convention (stated because implementations differ): the ground truth is partitioned
    into 26-connected components; for each component the prediction is restricted to that
    component's bounding box (optionally dilated by `bbox_dilate` voxels); a component
    with no predicted voxels at all scores 0.  Components smaller than `min_gt_voxels`
    are ignored.  Returns (mean, per_lesion_array); NaN if the ground truth has none.
    """
    g = np.isin(gt_seg, REGIONS[region])
    p = np.isin(pred_seg, REGIONS[region])
    lab, n = components(g)
    if n == 0:
        return float("nan"), np.zeros(0)
    scores = []
    for i in range(1, n + 1):
        comp = lab == i
        if comp.sum() < min_gt_voxels:
            continue
        sl = ndimage.find_objects(comp.astype(np.int32))[0]
        sl = tuple(slice(max(0, s.start - bbox_dilate), s.stop + bbox_dilate) for s in sl)
        pc = p[sl]
        scores.append(dice(pc.any() and (pc & comp[sl]), comp[sl]) if pc.any() else 0.0)
    scores = np.array(scores, dtype=np.float64)
    return (float(scores.mean()) if len(scores) else float("nan")), scores


def component_stats(pred_seg, gt_seg, region="WT"):
    """Component bookkeeping for the fragmentation story.

    returns dict(n_pred, n_gt, beta0_error=|n_pred-n_gt|, false_components, missed_lesions,
                 pred_voxels, gt_voxels)
    false_components : predicted components containing no ground-truth voxels
    missed_lesions   : ground-truth components containing no predicted voxels
    """
    p, g = np.isin(pred_seg, REGIONS[region]), np.isin(gt_seg, REGIONS[region])
    lp, npc = components(p)
    lg, ngc = components(g)
    false_c = 0
    for i in range(1, npc + 1):
        if not (g & (lp == i)).any():
            false_c += 1
    missed = 0
    for i in range(1, ngc + 1):
        if not (p & (lg == i)).any():
            missed += 1
    return dict(n_pred=npc, n_gt=ngc, beta0_error=abs(npc - ngc), false_components=false_c,
                missed_lesions=missed, pred_voxels=int(p.sum()), gt_voxels=int(g.sum()))


def volume_error(pred_seg, gt_seg, region="WT", voxel_vol_mm3=1.0):
    """(predicted - true) volume in mm^3 for one region of one case."""
    p = int(np.isin(pred_seg, REGIONS[region]).sum())
    g = int(np.isin(gt_seg, REGIONS[region]).sum())
    return float((p - g) * voxel_vol_mm3)


def bland_altman(errors):
    """Mean error and 95 % limits of agreement for an array of volume errors."""
    e = np.asarray(errors, dtype=np.float64)
    e = e[~np.isnan(e)]
    if len(e) == 0:
        return dict(n=0, bias=float("nan"), sd=float("nan"), lo=float("nan"), hi=float("nan"))
    bias, sd = float(e.mean()), float(e.std(ddof=1)) if len(e) > 1 else 0.0
    return dict(n=len(e), bias=bias, sd=sd, lo=bias - 1.96 * sd, hi=bias + 1.96 * sd)


# --------------------------------------------------------------------------------------
# self-tests -- no data, no GPU; run this file directly
# --------------------------------------------------------------------------------------

def _toy_case(seed=0, n=48):
    """A synthetic case: one big blobby lesion + optional small satellites."""
    rng = np.random.default_rng(seed)
    zz, yy, xx = np.mgrid[0:n, 0:n, 0:n].astype(np.float32)
    c = (n - 1) / 2.0
    big = (((xx - c) / 9) ** 2 + ((yy - c) / 7) ** 2 + ((zz - c) / 6) ** 2) <= 1
    small = (((xx - c - 15) / 2.5) ** 2 + ((yy - c + 12) / 2.5) ** 2 + ((zz - c) / 2.5) ** 2) <= 1
    tiny = (((xx - c + 17) / 1.5) ** 2 + ((yy - c - 14) / 1.5) ** 2 + ((zz - c + 8) / 1.5) ** 2) <= 1
    return big, small, tiny


def _test_postproc_operators():
    big, small, tiny = _toy_case()
    assert component_sizes(small)[0] > component_sizes(tiny)[0]
    # 1. filter removes exactly the small components, keeps the big one
    pred = (big | small | tiny)
    kept = filter_components(pred, int(component_sizes(small)[0]) + 1)
    assert np.array_equal(kept, big), "filter must keep only components >= threshold"
    # 2. monotonicity: raising the threshold never adds voxels
    for t in range(1, 400, 37):
        assert (filter_components(pred, t + 10) & ~filter_components(pred, t)).sum() == 0
    # 3. nested-region post-processing: the WT region must survive untouched, and a
    #    removed ET component must fall back to necrosis (never to a spurious label)
    seg = np.zeros_like(big, dtype=np.uint8)
    seg[big] = ED
    seg[small] = ET
    seg[tiny] = ET
    out = postproc_seg(seg, dict(min_vox={"WT": 1, "TC": 1,
                                          "ET": int(component_sizes(tiny)[0]) + 1},
                                 min_et_voxels=0))
    assert set(np.unique(out).tolist()) <= {0, NCR, ED, ET}, np.unique(out)
    assert np.isin(out, REGIONS["WT"]).sum() == np.isin(seg, REGIONS["WT"]).sum(), \
        "WT must be preserved when its own threshold is 1"
    assert (out[tiny] == NCR).all(), "the removed ET component must become necrosis"
    # 4. ET relabeling
    seg2 = np.zeros_like(big, dtype=np.uint8)
    seg2[big] = ED
    seg2[tiny] = ET
    out2 = relabel_et_to_necrosis(seg2, min_et_voxels=int(tiny.sum()) + 1)
    assert (out2 == ET).sum() == 0 and (out2 == NCR).sum() == int(tiny.sum()), "ET must become NCR"
    out3 = relabel_et_to_necrosis(seg2, min_et_voxels=1)
    assert (out3 == ET).sum() == int(tiny.sum()), "a large-enough ET must be kept"
    # 5. empty mask is handled everywhere
    z = np.zeros((8, 8, 8), bool)
    assert filter_components(z, 5).sum() == 0 and component_sizes(z).size == 0
    print("  [ok] post-processing operators (filter, monotonicity, nesting, ET relabel, empty)")


def _toy_probs(seed, n=28, quality=0.85):
    """GT = big|small; predicted probability = quality inside GT, small noise outside."""
    rng = np.random.default_rng(1000 + seed)
    big, small, _ = _toy_case(seed, n)
    gt = big | small
    p = rng.uniform(0.0, 0.35, size=(n, n, n)).astype(np.float32)
    p[gt] = np.clip(quality + rng.normal(0, 0.18, gt.sum()), 0, 1).astype(np.float32)
    return p, gt


def _test_crc_guarantee(n_splits=200, alpha=0.10, n_cal=37, n_test=37, pool_n=300):
    """The headline statistical property: averaged over calibration draws, the realised
    risk of the chosen threshold must not exceed alpha, and the certificate must be
    TIGHT (not the vacuous 'predict everything' solution)."""
    THR = np.linspace(0.0, 1.0, 201)
    pool = [_toy_probs(s) for s in range(pool_n)]
    rng = np.random.default_rng(0)
    realised, thresholds = [], []
    for k in range(n_splits):
        idx = rng.permutation(pool_n)
        cal = [pool[i] for i in idx[:n_cal]]
        tst = [pool[i] for i in idx[n_cal:n_cal + n_test]]
        R = np.stack([miss_curve(p, g, THR) for p, g in cal])
        d = crc_threshold(R, alpha=alpha, thresholds=THR)
        assert d["feasible"], "synthetic model should easily meet alpha=0.10"
        assert d["threshold"] > 0.05, "certificate must not degenerate to predict-everything"
        realised.append(np.mean([per_case_risk(p, g, d["threshold"]) for p, g in tst]))
        thresholds.append(d["threshold"])
    m = float(np.mean(realised))
    assert m <= alpha * 1.02, f"CRC violated: mean realised risk {m:.4f} > {alpha}"
    assert m >= alpha * 0.3, f"suspiciously conservative ({m:.4f}) -- check the formula"
    thr_sd = float(np.std(thresholds))
    print(f"       thresholds chosen: mean {np.mean(thresholds):.3f}, sd {thr_sd:.3f} "
          f"(calibration noise at n_cal={n_cal})")
    # alpha must actually matter: a tighter target yields a smaller realised risk
    R = np.stack([miss_curve(p, g, THR) for p, g in pool[:n_cal]])
    t_lo = crc_threshold(R, 0.05, thresholds=THR)["threshold"]
    t_hi = crc_threshold(R, 0.30, thresholds=THR)["threshold"]
    assert t_lo < t_hi, "smaller alpha must give a larger prediction set (lower threshold)"
    print(f"  [ok] CRC guarantee: mean realised risk {m:.4f} <= alpha {alpha} over "
          f"{n_splits} calibration draws (n_cal={n_cal}); alpha monotone")


def _test_envelope_axis():
    """Regression test for the axis bug described in `monotone_envelope`."""
    rows = np.array([[0.10, 0.20, 0.30],
                     [0.30, 0.10, 0.20]])          # second row has a dip
    up = monotone_envelope(rows, "increasing")
    assert np.allclose(up[0], [0.10, 0.20, 0.30]), up
    assert np.allclose(up[1], [0.30, 0.30, 0.30]), up
    assert up.max() <= 0.30 + 1e-12, "the envelope must not leak across calibration cases"
    down = monotone_envelope(rows, "decreasing")
    assert np.allclose(down[0], [0.30, 0.30, 0.30]) and np.allclose(down[1], [0.30, 0.20, 0.20])
    one_d = monotone_envelope(np.array([0.5, 0.2, 0.4]))
    assert np.allclose(one_d, [0.5, 0.5, 0.5]), one_d
    print("  [ok] monotone envelope acts on the threshold axis only (no cross-case leakage)")


def _test_volume_interval():
    """The two-sided certificate: both risks must hold on held-out data, and the implied
    volume interval must contain the truth for most cases."""
    THR = np.linspace(0.0, 1.0, 201)
    pool = [_toy_probs(s, quality=0.75) for s in range(200)]
    rng = np.random.default_rng(7)
    idx = rng.permutation(200)
    cal_i, tst_i = idx[:60], idx[60:160]
    FN = np.stack([monotone_envelope(miss_curve(pool[i][0], pool[i][1], THR)) for i in cal_i])
    FP = np.stack([monotone_envelope(fp_curve(pool[i][0], pool[i][1], THR), "decreasing")
                   for i in cal_i])
    for alpha, beta in ((0.05, 0.20), (0.10, 0.20), (0.15, 0.30)):
        d = certified_threshold_pair(FN, FP, THR, alpha=alpha, beta=beta)
        assert d["feasible"], f"targets {alpha}/{beta} should be reachable on this toy data"
        assert d["thr_fn"] >= d["thr_fp"], "the admissible band must be a real interval"
        r_fn, r_fp, inside = [], [], []
        for i in tst_i:
            p, g = pool[i]
            pred = p >= d["lam"]
            r_fn.append(float((g & ~pred).sum()) / max(1, int(g.sum())))
            r_fp.append(0.0 if pred.sum() == 0 else float((pred & ~g).sum()) / int(pred.sum()))
            lo, hi = volume_interval(float(pred.sum()), alpha, beta)
            inside.append(lo <= float(g.sum()) <= hi)
        m_fn, m_fp = float(np.mean(r_fn)), float(np.mean(r_fp))
        assert m_fn <= alpha + 0.03, f"missed-tumour certificate violated: {m_fn:.3f} > {alpha}"
        assert m_fp <= beta + 0.05, f"wrong-prediction certificate violated: {m_fp:.3f} > {beta}"
        cov = float(np.mean(inside))
        assert cov >= 0.75, f"volume interval contains the truth only {cov:.0%} of the time"
        print(f"       alpha={alpha:.2f} beta={beta:.2f} -> lambda*={d['lam']:.2f} "
              f"(band [{d['thr_fp']:.2f},{d['thr_fn']:.2f}]), realised FN {m_fn:.3f}, "
              f"FP {m_fp:.3f}, interval containment {cov:.0%}")
    # the two targets move the band in OPPOSITE directions -- that is the whole point:
    #   * a tighter missed-tumour target needs an EARLIER (lower) threshold -> bigger set
    #   * a tighter wrong-prediction target raises the FLOOR of the band
    d_a_lo = certified_threshold_pair(FN, FP, THR, alpha=0.05, beta=0.20)
    d_a_hi = certified_threshold_pair(FN, FP, THR, alpha=0.30, beta=0.20)
    assert d_a_lo["thr_fn"] <= d_a_hi["thr_fn"], (d_a_lo["thr_fn"], d_a_hi["thr_fn"])
    d_b_lo = certified_threshold_pair(FN, FP, THR, alpha=0.20, beta=0.05)
    d_b_hi = certified_threshold_pair(FN, FP, THR, alpha=0.20, beta=0.40)
    assert d_b_lo["thr_fp"] >= d_b_hi["thr_fp"], (d_b_lo["thr_fp"], d_b_hi["thr_fp"])
    print(f"       monotone: alpha 0.05->0.30 moves thr_fn {d_a_lo['thr_fn']:.2f}->"
          f"{d_a_hi['thr_fn']:.2f}; beta 0.05->0.40 moves thr_fp {d_b_lo['thr_fp']:.2f}->"
          f"{d_b_hi['thr_fp']:.2f}")
    print("  [ok] two-sided certificates + volume interval (both risks hold on held-out cases)")


def _test_stratified():
    """Low-quality cases must get their own (more conservative) threshold."""
    THR = np.linspace(0, 1, 201)
    easy = [_toy_probs(s, quality=0.95) for s in range(30)]
    hard = [_toy_probs(s + 500, quality=0.60) for s in range(30)]
    R_easy = np.stack([miss_curve(p, g, THR) for p, g in easy])
    R_hard = np.stack([miss_curve(p, g, THR) for p, g in hard])
    res = stratified_crc({"low_contrast": R_hard, "high_contrast": R_easy}, alpha=0.10,
                         thresholds=THR)
    assert res["low_contrast"]["threshold"] < res["high_contrast"]["threshold"], \
        "the harder stratum must receive the lower (more inclusive) threshold"
    pooled = res["__pooled__"]["threshold"]
    assert res["low_contrast"]["threshold"] <= pooled, "pooling hides the hard stratum"
    print(f"  [ok] stratified CRC: hard stratum threshold {res['low_contrast']['threshold']:.2f} "
          f"< easy {res['high_contrast']['threshold']:.2f} (pooled {pooled:.2f})")


def _test_certified_filter():
    """A filter that deletes true satellites must be bounded by alpha, and the bound must
    stop it from ever deleting the essential (large) lesion."""
    big_size = int(_toy_case(0)[0].sum())
    THR = np.array([1, 10, 25, 50, 100, 200, 400, 800, 1600, 2400])
    cal, tst = [], []
    for s in range(60):
        big, small, tiny = _toy_case(s)
        gt = (big | small | tiny).astype(np.uint8)
        pred = (big | small | tiny).astype(np.uint8)
        if s % 3 == 0:                        # the model sometimes misses the satellite
            pred[small] = 0
        (cal if s < 40 else tst).append(removal_curve(pred, gt, THR, region="WT"))
    d = certified_filter_threshold(np.stack(cal), alpha=0.10, thresholds=THR)
    assert d["feasible"], "a modest filter must be certifiable at alpha=0.10"
    j = int(np.where(THR == d["threshold"])[0][0])
    realised = float(np.mean([r[j] for r in tst]))
    # 1. the guarantee holds on held-out cases
    assert realised <= 0.10 + 1e-9, realised
    # 2. the certificate protects essential anatomy: it must never pick a threshold that
    #    would delete the big lesion
    assert d["threshold"] < big_size, (d["threshold"], big_size)
    # 3. a naive fixed threshold is NOT certified -> the certificate added something
    RC = np.stack(cal)
    jbad = int(np.where(THR == 2400)[0][0])          # 2400 > the big lesion's size
    bound_bad = (RC.shape[0] / (RC.shape[0] + 1.0)) * RC[:, jbad].mean() + 1.0 / (RC.shape[0] + 1.0)
    assert bound_bad > 0.10, f"a 2400-voxel filter must fail the bound (got {bound_bad:.2f})"
    # 4. damage is charged to the filter, not to the model: a perfectly missed satellite
    #    must not appear as filter damage
    gt_only = (big | small | tiny).astype(np.uint8)
    missed = np.zeros_like(gt_only); missed[big] = 2
    r = removal_curve(missed, gt_only, np.array([50.0]), region="WT")
    assert r[0] == 0.0, "voxels the model never predicted must not count as filter damage"
    # 5. the engineering floor caps a statistically valid but unnerving threshold
    d_cap = certified_filter_threshold(np.stack(cal), alpha=0.10, thresholds=THR,
                                       component_size_floor=200.0)
    assert d_cap["capped"] and d_cap["threshold"] == 200.0 and d_cap["certified_threshold"] == 1600.0, d_cap
    assert d_cap["threshold"] < big_size
    _c = dict(lam=0.42, n_cal=37, curves=dict(fn=[0.3], fp=[0.1]),
              strata=dict(cut=11.4, n_low=12, n_high=25, lam_low=0.34, lam_high=0.46,
                          feasible_low=True, feasible_high=True,
                          curves_low=dict(fn=[0.2], fp=[0.08]),
                          curves_high=dict(fn=[0.1], fp=[0.05])))
    assert choose_certificate(_c, cnr=9.0)["lam"] == 0.34
    assert choose_certificate(_c, cnr=9.0)["scope"] == "low"
    assert choose_certificate(_c, cnr=15.0)["lam"] == 0.46
    assert choose_certificate(_c, cnr=15.0)["n_cal"] == 25
    assert choose_certificate(_c, cnr=np.nan)["scope"] == "pooled"
    assert choose_certificate(_c, cnr=9.0, stratified=False)["lam"] == 0.42
    _c["strata"]["feasible_low"] = False           # never hand back a dead stratum
    assert choose_certificate(_c, cnr=9.0)["scope"] == "pooled"
    assert choose_certificate(dict(lam=0.5, n_cal=10), cnr=9.0)["scope"] == "pooled"
    print("[ok] choose_certificate: stratum when certifiable, pooled otherwise")

    print(f"       floor: certified {int(d_cap['certified_threshold'])} capped to "
          f"{int(d_cap['threshold'])} by the observed lesion-size floor")
    print(f"  [ok] certified filter: delta={int(d['threshold'])} voxels (< big lesion "
          f"{big_size}), cal bound {d['bound_cal']:.3f}, realised collateral {realised:.3f} "
          f"<= 0.10; a naive delta=2400 is rejected (bound {bound_bad:.2f})")


def _test_metrics():
    big, small, tiny = _toy_case()
    gt = np.zeros(big.shape, np.uint8)
    gt[big] = ED
    gt[small] = ET
    gt[tiny] = ET
    pred = np.zeros_like(gt)
    pred[big] = ED
    pred[small] = ET                      # tiny is missed entirely
    # lesion-wise dice: ET has 2 GT components, one predicted, one missed -> mean in (0,1)
    m, per = lesionwise_dice(pred, gt, region="ET")
    assert len(per) == 2 and 0.0 <= m <= 1.0 and per.min() == 0.0, (m, per)
    st = component_stats(pred, gt, region="ET")
    assert st["missed_lesions"] == 1 and st["false_components"] == 0, st
    # a spurious island must show up as a false component (that is the whole point of the
    # fragmentation metric), while a missed lesion must be counted separately
    pred2 = pred.copy()
    pred2[2:5, 2:5, 2:5] = ET             # spurious blob in the corner, no GT there
    st2 = component_stats(pred2, gt, region="ET")
    assert st2["false_components"] == 1 and st2["missed_lesions"] == 1, st2
    assert st2["n_pred"] == 2 and st2["n_gt"] == 2 and st2["beta0_error"] == 0, st2
    # WT: three disconnected GT blobs, two of them predicted
    st3 = component_stats(pred, gt, region="WT")
    assert st3["n_gt"] == 3 and st3["n_pred"] == 2 and st3["missed_lesions"] == 1, st3
    assert st3["beta0_error"] == 1, st3
    d = dice_regions(pred, gt)
    assert d["ET"] < 1.0 and d["WT"] > 0.5, d
    # HD95 is NaN when one side is empty, and finite otherwise
    assert not np.isnan(hd95(pred, pred, "WT")), "identical masks must give HD95 = 0"
    empty = np.zeros_like(pred)
    assert np.isnan(hd95(empty, gt, "ET")), "undefined HD95 must be NaN, not 0"
    ve = volume_error(pred, gt, "WT")
    assert ve < 0, "this prediction misses a lesion, so the volume error must be negative"
    ba = bland_altman([ve, ve])
    assert ba["n"] == 2 and ba["sd"] == 0.0 and ba["bias"] == ve, ba
    print("  [ok] metrics: lesion-wise Dice, component stats, Dice, HD95 conventions, Bland-Altman")


def _selftest():
    print("safepp self-test")
    _test_postproc_operators()
    _test_metrics()
    _test_envelope_axis()
    _test_crc_guarantee()
    _test_volume_interval()
    _test_stratified()
    _test_certified_filter()
    print("all self-tests passed")


# (self-tests disabled inside the notebook; run code/safepp.py standalone)

# ============================================================================
# F. v2 additions: the structure-recall target and its patch sampler
# ============================================================================
class SkelPatchDataset(PatchDataset):
    """PatchDataset + the thin ground-truth target of the three regions.

    N0 precomputes {pid}_skel.npz (WT/TC/ET thin targets, see skeletonize_mask and
    component_recall_loss) once; training then never has to thin anything on the GPU.  Sample
    i is chosen EXACTLY as in PatchDataset (same seed, same foreground centre, same flips), so
    the target patch is voxel-aligned with the label patch -- `selftest_skel_dataset()`
    verifies that alignment (kept deliberately strict).
    """

    def __init__(self, cache, ids, length, skel_root=None, **kw):
        super().__init__(cache, ids, "base", length, **kw)   # arm is unused here
        self.skel_root = as_roots(skel_root) or self.cache

    def __getitem__(self, i):
        rng = np.random.default_rng([self.seed, self.offset + i])
        pid = self.ids[int(rng.integers(len(self.ids)))]
        mri = np.load(self._p(pid, "mri"), mmap_mode="r")
        seg = np.load(self._p(pid, "seg"), mmap_mode="r")
        dist = np.load(self._p(pid, "dist"), mmap_mode="r")
        shape = seg.shape

        center = None
        if rng.random() < self.P_FG:
            locs = np.load(self._p(pid, "locs", "npz"))
            present = [k for k in ("c1", "c2", "c4") if len(locs[k]) > 0]
            if present:
                k = present[int(rng.integers(len(present)))]
                center = locs[k][int(rng.integers(len(locs[k])))]
        if center is None:
            center = [rng.integers(s) for s in shape]
        st = [int(np.clip(c - p // 2, 0, max(0, s - p))) for c, s, p in zip(center, shape, self.patch)]
        sl = tuple(slice(a, a + p) for a, p in zip(st, self.patch))

        x = np.asarray(mri[(slice(None),) + sl], dtype=np.float32)
        y = seg_to_targets(np.asarray(seg[sl]))
        dd = decode_dist(np.asarray(dist[sl]))[None]
        f = _first_file(self.skel_root, f"{pid}_skel.npz")
        with np.load(f) as z:
            sk = np.stack([np.asarray(z[k][sl], np.float32) for k in ("wt", "tc", "et")])

        pad = [(0, 0)] + [(0, p - s) for p, s in zip(self.patch, x.shape[1:])]
        if any(b for _, b in pad):
            x = np.pad(x, pad); y = np.pad(y, [(0, 0)] + pad[1:])
            dd = np.pad(dd, [(0, 0)] + pad[1:]); sk = np.pad(sk, [(0, 0)] + pad[1:])
        for ax in (1, 2, 3):
            if rng.random() < 0.5:
                x, y, dd, sk = (np.flip(a, ax) for a in (x, y, dd, sk))
                break
        # (a single flip pass, exactly as PatchDataset does; the generator above keeps the
        #  RNG stream identical to the parent because all four arrays are flipped together)
        for c in range(min(4, x.shape[0])):
            if rng.random() < 0.5:
                m = x[c] != 0
                x[c] = np.where(m, x[c] * rng.uniform(0.9, 1.1) + rng.uniform(-0.1, 0.1), 0.0)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(y)),
                torch.from_numpy(np.ascontiguousarray(dd)),
                torch.from_numpy(np.ascontiguousarray(sk)))


def component_recall_loss(prob, target, eps=1e-6):
    """Soft recall of the PREDICTION on a thin GROUND-TRUTH target (the lesion's core).

        L = 1 - sum(p * T) / sum(T)

    `target` is `centre_ridge(mask)`: the Euclidean-distance ridge of the mask, computed per
    26-connected component (see N0 / `build_skel_*`).  What that buys, in one sentence per
    case:

      * elongated structure (enhancing rim, vessel-like ET, a thin edema arm) -> the ridge is
        a CURVE along its middle, so the term demands that the prediction cover the whole
        structure, not just its fat end;
      * a compact lesion -> the ridge is a short core near its centre;
      * every ground-truth COMPONENT gets at least one ridge voxel, so the term directly
        penalises MISSING A SATELLITE LESION -- a beta_0 (component-count) recall, which is
        the failure mode this project actually has.

    HONEST SCOPE: this is NOT the published Skeleton Recall Loss (ECCV 2024, arXiv
    2404.03010) and must not be quoted as it.  The published method thins the GT with Lee's
    algorithm into a homotopy-equivalent skeleton; the ridge used here is thinner-and-centred
    but not homotopy-preserving.  The FORM is identical (one overlap term, no differentiable
    skeletonisation, a few tensor ops), and `SKEL_BACKEND='skimage'` swaps the target for
    Lee's thinning when that build passes `_skimage_skeleton_ok()` -- i.e. you can run the
    published variant from the same code, and N0 prints which target it built.

    `prob` and `target` are (B, C, D, H, W); the reduction is over voxels and classes.
    """
    # the sums ask for fp32 explicitly: `torch.sum` on a half tensor returns a half, and a
    # whole-patch sum of fp16 values overflows 65504 -> inf -> nan (exactly the Dice problem)
    num = (prob * target).sum(dtype=torch.float32)
    den = target.sum(dtype=torch.float32) + eps
    return 1.0 - num / den


# old name kept so nothing that already imports it breaks
skeleton_recall_loss = component_recall_loss


class DiceLossFast(nn.Module):
    """MONAI `DiceLoss(sigmoid=True, squared_pred=True, smooth_nr=0.0, smooth_dr=1e-5, batch=True)`
    reimplemented so the ELEMENTWISE maths stays in the autocast dtype (fp16 on GPU = fast) while
    every REDUCTION accumulates in fp32.  Verified against MONAI in fp32 (same value to ~1e-5,
    `code/test_dice_fast.py`).

    Why this exists -- it is not a micro-optimisation, it is the bug that killed two GPU runs:
    the reduction in MONAI's DiceLoss is `torch.sum(half_tensor)`, which ACCUMULATES in fp32 but
    RETURNS A HALF.  Over a 96^3 patch that result is ~1.8e5, i.e. above fp16's 65504 -> inf ->
    nan loss -> nan weights -> a model that predicts nothing for the whole run (`loss nan` for
    8000 iterations, `best=0.0556`, in both arms).  Casting the whole loss to fp32 fixes it but
    makes the elementwise ops (and their backward) fp32 as well, measured ~5x slower per
    iteration on the user's T4 (0.33 -> 1.6 s/it).  Here the sums ask for fp32 explicitly:
    `(p * p).sum(dtype=torch.float32)`, so nothing overflows and nothing is wasted.
    """

    def __init__(self, smooth_nr=0.0, smooth_dr=1e-5, batch=True):
        super().__init__()
        self.smooth_nr, self.smooth_dr, self.batch = float(smooth_nr), float(smooth_dr), bool(batch)

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)                       # autocast dtype: fp16 on GPU
        y = targets if targets.dtype == p.dtype else targets.to(p.dtype)
        dims = list(range(2, p.ndim)) + ([0] if self.batch else [])
        # MONAI's order and formula, so the value matches to rounding
        intersection = (p * y).sum(dim=dims, dtype=torch.float32)
        ground_o = (y * y).sum(dim=dims, dtype=torch.float32)
        pred_o = (p * p).sum(dim=dims, dtype=torch.float32)
        f = 1.0 - (2.0 * intersection + self.smooth_nr) / (ground_o + pred_o + self.smooth_dr)
        return f.mean()


class TopologyLoss(nn.Module):
    """dice+BCE plus an optional structure-recall term whose strength is SCHEDULED by the
    training loop (`self.w` is a plain float, ramped from 0 to the arm's nominal value).

    Why not a learnable multiplier -- measured, not assumed: with `w` as an nn.Parameter the
    gradient of the loss w.r.t. w is +R(theta) > 0 at whatever theta the model currently has,
    so plain gradient descent drives w below zero, relu(w) kills the term and the arm becomes
    silently identical to its control (observed: w = -0.006 after 25 iterations, i.e. off).
    A learnable weight can only earn its place through a meta-gradient over the whole training
    run, which a single scalar under SGD does not have.

    What is kept is the property that actually matters: w starts at ZERO, so step 0 and the
    whole warm-up phase are bit-identical to the Dice+BCE control, and whether the term earns
    its place is decided by the held-out metrics (N2/N3), not by the scalar itself.
    """

    def __init__(self, w0=0.0, boundary=True):
        super().__init__()
        self.dice = DiceLossFast()      # fp16-safe + fast; same value as MONAI's DiceLoss
        self.boundary = boundary
        self.w = float(w0)                       # ramped by the training loop; not a parameter
        self.nominal = float(w0)

    def forward(self, logits, y, dist=None, skel=None):
        loss = self.dice(logits, y)
        loss = loss + (distance_weighted_bce(logits, y, dist)
                       if (dist is not None and self.boundary)
                       else F.binary_cross_entropy_with_logits(logits, y))
        if skel is not None and float(self.w) > 0:
            loss = loss + float(self.w) * component_recall_loss(torch.sigmoid(logits), skel)
        return loss


def _skimage_skeleton_ok():
    """Is this build's skimage.skeletonize trustworthy?

    NOT a formality: builds exist that return an ALL-ZERO skeleton for shapes as simple as a
    solid cube (observed in the wild with scikit-image 0.25.x).  A silently empty skeleton
    would make the topology arm train on nothing, so we detect it and fall back.
    Returns (ok, note).
    """
    try:
        from skimage.morphology import skeletonize
    except Exception as e:                                   # pragma: no cover
        return False, f"skimage unavailable ({e})"
    zz, yy, xx = np.mgrid[0:32, 0:32, 0:32]
    probes = (("cube8", np.ones((8, 8, 8), bool), 4),
              ("tube4x4x17", _probe_tube(), 8),
              ("ellipsoid5x6x7", (((zz - 16) / 5.0) ** 2 + ((yy - 16) / 6.0) ** 2
                                  + ((xx - 16) / 7.0) ** 2) <= 1, 4))
    notes = []
    for name, m, floor in probes:
        try:
            n = int(np.asarray(skeletonize(m.astype(bool))).sum())
        except Exception as e:                               # pragma: no cover
            return False, f"skeletonize raised on {name} ({e})"
        if n < floor:
            return False, (f"skeletonize is degenerate here: {name} ({int(m.sum())} voxels "
                           f"in) -> {n} skeleton voxel(s), expected >= {floor}")
        notes.append(f"{name}={n}")
    return True, "ok (" + ", ".join(notes) + ")"


def _probe_tube():
    t = np.zeros((10, 10, 20), bool)
    t[3:7, 3:7, 2:19] = True
    return t


def centre_ridge(mask):
    """Dependency-free centre line: the ridge of the Euclidean distance transform.

    Voxels that sit inside the mask and are a local maximum of the distance to the background
    in their 26-neighbourhood.  For a tube that is a curve, for a blob a medial sheet -- both
    are thin and centred, which is what the skeleton-recall term needs (it asks how much of
    the centre of the lesion the prediction covers).

    Guarantee: the result is a subset of the mask and is never empty when the mask is not.
    It is NOT a homotopy-equivalent thinning; `skeletonize_mask` prefers Lee thinning when
    that is available and trustworthy.
    """
    from scipy import ndimage
    m = np.asarray(mask, bool)
    if not m.any():
        return m.copy()
    # pad by one background voxel: without it the distance transform of a mask that touches
    # the array border measures the distance to the array corner instead of to the boundary
    mp = np.pad(m, 1)
    d = ndimage.distance_transform_edt(mp)
    ridge = mp & (d >= ndimage.maximum_filter(d, size=3))     # local maxima of the distance
    if not ridge.any():                                       # single-voxel mask, etc.
        ridge = mp & (d == d.max())
    lab, n = ndimage.label(mp, structure=np.ones((3, 3, 3), int))
    for i in range(1, n + 1):                                 # every component represented
        comp = lab == i
        if not (ridge & comp).any():
            ridge |= comp & (d == d[comp].max())
    return ridge[1:-1, 1:-1, 1:-1]


def skeletonize_mask(mask, backend="auto", verbose=False):
    """Thin, centred skeleton of `mask`, with a self-checked backend.

    backend='auto'             skimage's 3-D Lee thinning if that build passes a sanity check,
                               otherwise the dependency-free distance-transform ridge
    backend='skimage'|'ridge'  force one
    """
    m = np.asarray(mask, bool)
    if not m.any():
        return m.copy()
    if backend == "auto":
        ok, note = _skimage_skeleton_ok()
        backend = "skimage" if ok else "ridge"
        if verbose:
            print(f"  skeleton backend: {backend} ({note})")
    if backend == "skimage":
        from skimage.morphology import skeletonize
        out = np.asarray(skeletonize(m.astype(bool))).astype(bool)
        if out.any():
            return out
    return centre_ridge(m)


def skeleton_selftest(verbose=True):
    """Fail loudly if the skeletons the topology arm will train on are degenerate."""
    zz, yy, xx = np.mgrid[0:24, 0:24, 0:24]
    blob = (((zz - 11) / 5.0) ** 2 + ((yy - 11) / 6.0) ** 2 + ((xx - 11) / 7.0) ** 2) <= 1
    tube = np.zeros((24, 24, 24), bool)
    tube[10:14, 10:14, 3:21] = True
    out = {}
    for name, m in (("blob", blob), ("tube", tube), ("blob+tube", blob | tube),
                    ("cube8", np.ones((8, 8, 8), bool))):
        sk = skeletonize_mask(m)
        assert sk.any(), f"{name}: empty skeleton for a non-empty mask"
        assert not (sk & ~m).any(), f"{name}: skeleton outside the mask"
        frac = sk.sum() / max(1, int(m.sum()))
        assert frac < 0.9, f"{name}: skeleton is not thinner than the mask ({frac:.2f})"
        out[name] = (int(m.sum()), int(sk.sum()))
    ok, note = _skimage_skeleton_ok()
    if verbose:
        print(f"[ok] skeletonize_mask {out} | skimage backend usable: {ok} ({note})")
    return out


def selftest_skel_dataset(cache, ids, patch=(32, 32, 32), n=8, seed=42, skel_root=None):
    """Verify the skeleton patch is voxel-aligned with the targets (skel[0] must be the WT
    skeleton, i.e. nonzero exactly where y[0] is nonzero for a non-empty target patch)."""
    a = PatchDataset(cache, ids, "base", length=n, patch=patch, seed=seed, dod_tag=None)
    b = SkelPatchDataset(cache, ids, length=n, patch=patch, seed=seed, skel_root=skel_root)
    for i in range(n):
        x1, y1, d1 = a[i]
        x2, y2, d2, s = b[i]
        assert torch.equal(y1, y2), f"targets differ at sample {i}"
        assert torch.equal(d1, d2), f"distance maps differ at sample {i}"
        assert s.shape == y1.shape, (s.shape, y1.shape)
        # WT skeleton must sit inside the WT target (allowing the 1-voxel skeleton margin)
        assert float((s[0] * (1 - y1[0])).sum()) <= 0.02 * float(y1[0].sum()) + 5, \
            f"skeleton not aligned with the target at sample {i}"
    return True


# ---- aliases: the cells below say bratsx.x() / safepp.x() -------------------
# In a notebook this cell runs in __main__, so both names are simply THIS
# namespace: nothing to import, nothing to attach, and a forked DataLoader
# worker resolves them for free.  (The fallback covers the case where the
# library is exec'd somewhere other than __main__, e.g. a test harness.)
import sys as _sys, types as _types
_ns = globals()
bratsx = _sys.modules.get(__name__)
if getattr(bratsx, "__dict__", None) is not _ns:
    bratsx = _types.ModuleType("bratsx")
    bratsx.__dict__.update(_ns)
safepp = bratsx
print("library ready:", len([k for k in dir(bratsx) if not k.startswith('_')]),
      "names (bratsx and safepp are aliases of this notebook's namespace)")


# %% [CELL 4] ENVIRONMENT ---------------------------------------------------
import gc, glob, json, os, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

import importlib.util as _iu
_missing = [m for m in ("torch", "monai", "numpy", "scipy", "tqdm", "pandas", "skimage")
            if _iu.find_spec(m) is None]
if _missing:
    raise SystemExit(f"missing package(s): {_missing}. Turn Internet ON and run "
                     f"!pip install {' '.join(_missing)}  (or add them in the Dependency "
                     f"Manager), then re-run from the top.")

os.makedirs(WORK, exist_ok=True)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print("GPU:", torch.cuda.get_device_name(0),
          "| mem", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("!! no GPU: this notebook is meant for a GPU session "
          "(Settings -> Accelerator -> GPU P100 or T4 x2)")

if ADOPT_FROM:
    bratsx.adopt_previous_runs(ADOPT_FROM, WORK)
RUNS = os.path.join(WORK, "runs"); os.makedirs(RUNS, exist_ok=True)


# fp16 autocast: the forward/backward run in fp16 on CUDA, but the LOSS is always fp32 (below).
USE_AMP = (device.type == "cuda") if AMP is None else bool(AMP)
print(f"autocast: {USE_AMP} (fp16 forward, fp16 elementwise in the loss) | loss reductions in "
      f"fp32 (a fp16 whole-patch sum overflows 65504 -> nan: that is what killed two runs)")

def make_scaler():
    """AMP gradient scaler -- two jobs, both needed:

      * skip the optimizer step when the GRADIENTS are non-finite, instead of writing inf/nan into
        the fp32 weights.  Once a weight is nan the model predicts nan forever, which is exactly
        what ended the sr50 arm at iteration 1103 (loss nan although the inputs were finite, i.e.
        the weights were already poisoned);
      * scale the loss so small fp16 gradients do not underflow to zero.

    Without this, autocast is a half-finished recipe: nothing stands between one bad gradient and a
    dead run.  On CPU, or with AMP off, it is an inert passthrough.
    """
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except (AttributeError, TypeError):          # older torch: legacy signature
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


SCALER = make_scaler()



def _find_all(tag, probe):
    hits = [p for p in glob.glob(os.path.join(INPUT_ROOT, "**", "*"), recursive=True)
            if os.path.isdir(p) and os.path.basename(p).startswith(tag)
            and glob.glob(os.path.join(p, probe))]
    return sorted(set(hits))


CACHE_BASE = bratsx.as_roots(CACHE_BASE) or _find_all("cache_base", "*_mri.npy") \
    or bratsx.as_roots(os.path.join(WORK, "cache_base"))
SKEL_ROOT = bratsx.as_roots(SKEL_ROOT) or _find_all("skel", "*_skel.npz") \
    or bratsx.as_roots(os.path.join(WORK, "skel"))
assert CACHE_BASE, f"cache not found under {INPUT_ROOT}: attach the N0 output dataset(s)"
print(f"cache: {len(CACHE_BASE)} shard(s) {CACHE_BASE}")
print(f"skel:  {len(SKEL_ROOT)} shard(s) {SKEL_ROOT if SKEL_ROOT else '(none)'}")

_sp = next((q for q in sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "dataset_split.json"),
                                        recursive=True)) + [os.path.join(WORK, "dataset_split.json")]
            if os.path.exists(q)), None)
assert _sp, "dataset_split.json not found: attach the cache dataset"
SPLIT = json.load(open(_sp))
TRAIN_IDS = [p for p in SPLIT["train"] if bratsx.has_case(CACHE_BASE, p)]
VAL_IDS = [p for p in SPLIT["val"] if bratsx.has_case(CACHE_BASE, p)]
VAL_SUB = VAL_IDS[:VAL_SUBSET]
print(f"split: train={len(TRAIN_IDS)} val={len(VAL_IDS)} (in-training subset {len(VAL_SUB)})")
assert TRAIN_IDS, "no training cases found -- are the cache shards attached?"

# the skeleton patch sampler must line up with the targets; fail loudly if it does not
if SKEL_ROOT:
    for _a in ("wt", "tc", "et"):
        _f = bratsx.cache_file(SKEL_ROOT, TRAIN_IDS[0], "skel", ext="npz")
        if not os.path.exists(_f):
            print(f"!! no skeleton file for {TRAIN_IDS[0]} -> the topology arm will not run")
            SKEL_ROOT = []
            break
if SKEL_ROOT:
    assert bratsx.selftest_skel_dataset(CACHE_BASE, TRAIN_IDS[:2], patch=(32, 32, 32),
                                        n=2, seed=SEED, skel_root=SKEL_ROOT)
    print("skeleton patches are voxel-aligned with the targets (checked)")


# %% [CELL 5] MODEL + LOSS --------------------------------------------------
def build(seed=SEED):
    torch.manual_seed(seed); np.random.seed(seed)
    model = bratsx._segresnet().to(device)        # MONAI SegResNet, 4ch in, 3 regions out
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    return model, opt


def arm_weight(arm):
    """Nominal strength encoded in the arm name: 'sr50' -> 0.5, 'sr25' -> 0.25."""
    if not arm.startswith("sr"):
        return 0.0
    if len(arm) > 2 and arm[2:].isdigit():
        return float(arm[2:]) / 100.0
    return {"sr25": 0.25, "sr50": 0.50, "sr100": 1.00}.get(arm, float(W_TOPOLOGY))


def make_loss(arm):
    """'base' -> Dice+BCE.  'srW' -> the same plus structure recall (the per-component
    centre line of the ground truth, see bratsx.component_recall_loss) at strength W."""
    crit = bratsx.TopologyLoss(w0=0.0).to(device)      # starts at 0: see TopologyLoss
    crit.nominal = arm_weight(arm)
    return crit


def schedule_w(crit, it, start_it):
    """w = nominal * min(1, (it - start_it) / WARMUP).  0 at the first step, by construction."""
    frac = (it - start_it) / max(1.0, float(W_TOPOLOGY_WARMUP))
    crit.w = crit.nominal * min(1.0, max(0.0, frac))
    return crit.w


def loss_forward(crit, logits, y, dist, skel, arm):
    """The strength of the recall term is the SCHEDULED value on `crit.w` (see schedule_w).

    PRECISION, and why it is arranged this way:
      * the Dice reduction is `torch.sum(half)` -> returns a HALF -> a whole-patch sum (~1.8e5)
        overflows fp16's 65504 -> inf -> nan -> every weight nan -> the model predicts nothing
        (`loss nan` for 8000 iterations, `best=0.0556`).  `crit.dice` is `bratsx.DiceLossFast`,
        which asks for fp32 sums while keeping the elementwise maths in the autocast dtype.
      * the topology term's sums are fp32 for the same reason (see component_recall_loss).
      * the distance-weighted BCE is a MEAN, so it cannot overflow -- but on CPU half ops are
        patchy, so the whole loss runs in fp32 there (no speed to win on CPU anyway).
    """
    if logits.device.type == "cpu":
        logits = logits.float()
        y = y.float()
        dist = None if dist is None else dist.float()
        skel = None if skel is None else skel.float()
    if arm.startswith("sr") and skel is not None:
        loss = crit.dice(logits, y)
        loss = loss + bratsx.distance_weighted_bce(logits, y, dist)
        if crit.w > 0:
            loss = loss + float(crit.w) * bratsx.component_recall_loss(torch.sigmoid(logits), skel)
        return loss
    return crit(logits, y, dist, None)


@torch.no_grad()
def validate(model, crit, ids, max_iters, it, patch=PATCH, overlap=0.25):
    """Fast check on a small validation subset: the same three region Dice the paper table
    uses.  Returns NaN when there is nothing to validate on."""
    if not ids:
        return float("nan"), {}
    model.eval()
    per_case = {}
    for pid in ids[:VAL_SUBSET]:
        x, seg, _ = bratsx.loader_case(CACHE_BASE, pid)
        probs = bratsx.infer_probs(model, x, patch, overlap, device, amp=USE_AMP)
        pred = probs >= 0.5
        gt = bratsx.seg_to_targets(seg) > 0.5
        per_case[pid] = {name: bratsx.dice_np(pred[i], gt[i]) for i, name in enumerate(("WT", "TC", "ET"))}
    model.train()
    mean = float(np.mean([np.mean(list(v.values())) for v in per_case.values()]))
    # a model that predicts NOTHING scores this on any cohort with an empty region: say so
    # out loud instead of leaving a number that looks like a bad-but-real result
    _empty = sum(1 for v in per_case.values() if all(x == 0.0 for x in v.values()))
    if _empty == len(per_case) and len(per_case) >= 2:
        print(f"!! the model predicts NO foreground anywhere ({len(per_case)} cases checked) -- "
              f"this is a failed run, not a weak one.  Look at the loss line above: if it is nan, "
              f"the loss overflowed (fp16 Dice sums); a constant loss with finite values means the "
              f"gradient never reached the model.")
    return mean, per_case


# %% [CELL 6] PILOT: how long is an iteration, and how many can we afford? ----
def run_dir(arm, seed):
    return os.path.join(RUNS, f"{arm}_seed{seed}")


def make_ds(arm, offset, length):
    """The training dataset.  `length` must cover the whole run: sample i is a pure function of
    (SEED+seed, offset+i), so a dataset that is SHORTER than the run re-serves the same patches on
    every pass -- a 96^3 patch is a small part of a real BraTS crop, and the model then trains on
    a handful of fixed patches and never learns (on the small synthetic fixture, where a patch
    covers a whole case, this bug is invisible).  offset = iterations already done, so a resumed
    run continues the sequence instead of repeating it."""
    use_skel = bool(SKEL_ROOT) and arm.startswith("sr")
    if use_skel:
        return bratsx.SkelPatchDataset(CACHE_BASE, TRAIN_IDS, length=length, patch=PATCH,
                                       seed=SEED + 0, offset=offset, skel_root=SKEL_ROOT)
    return bratsx.PatchDataset(CACHE_BASE, TRAIN_IDS, None, length=length, patch=PATCH,
                               seed=SEED + 0, offset=offset, dod_tag=None)


def _poisoned(rd):
    """The DONE payload of a run that predicts nothing, or None.

    This is the fp16-Dice failure signature: `loss nan` for the whole run, so every weight is nan
    and the model predicts nothing.  On a 12-case validation subset with 3 regions the score comes
    out at exactly 2/(12*3) = 0.0556 (dice_np scores an empty GT with an empty prediction 1.0).
    Such a checkpoint is dead: N2 would load it and certify nothing.
    """
    f = os.path.join(rd, "DONE")
    if not os.path.exists(f):
        return None
    try:
        d = json.load(open(f))
        if int(d.get("iters", 0)) <= 0:
            return None
        if not VAL_SUB:
            return None      # the split has no validation cases: there is nothing to judge from.
                             # Without this the "predicts nothing" test would flag EVERY run and
                             # the queue would retrain the same two arms in every session forever.
        best = float(d.get("best", 1.0))
        if best < 0:
            return None      # validation never ran in that run -- no evidence either way
        return d if best <= FAILED_BEST else None
    except Exception:
        return None


def _nan_weights(model):
    """True when a parameter already holds nan/inf -- a poisoned checkpoint or a poisoned model.

    Rolling BACK to such a checkpoint cannot help: every forward pass from it is nan, and the next
    rollback reloads the very same file.  Quarantine it instead (see train_one).
    """
    return any(bool((~torch.isfinite(p)).any()) for p in model.parameters())


def _quarantine(rd):
    """Move a run's checkpoints/records aside, so the next attempt starts clean but nothing is
    destroyed (the files stay on disk with a .failed suffix)."""
    moved = []
    for f in ("last.pt", "best.pt", "history.json", "train_log.csv", "val_last.csv", "DONE"):
        q = os.path.join(rd, f)
        if os.path.exists(q):
            os.replace(q, q + ".failed")
            moved.append(f)
    return moved


def is_done(arm, seed):
    """A finished run is skipped -- unless EXTEND is on and it has not reached TARGET_ITERS yet,
    in which case the next session continues it from last.pt.  A run that predicts nothing is
    never treated as done (see _poisoned)."""
    rd = run_dir(arm, seed)
    f = os.path.join(rd, "DONE")
    if not os.path.exists(f):
        return False
    bad = _poisoned(rd)
    if bad is not None and IGNORE_FAILED_RUNS:
        return False          # train_one() explains and moves the dead attempt aside
    if not EXTEND:
        return True
    try:
        d = json.load(open(f))
    except Exception:
        return True
    if float(d.get("best", -1)) < 0:
        print(f"[{arm}/{seed}] note: this DONE has no validation score (best={d.get('best')}) -- "
              f"{'the split has no validation cases' if not VAL_SUB else 'validation never ran'}. "
              f"N2 can still evaluate it, but there is no honest in-training signal.  "
              f"Do not read a curve into it.")
    return int(d.get("iters", 0)) >= min(TARGET_ITERS, MAX_ITERS_CAP)


def pilot(arm="base"):
    """How many iterations does one RUN_TARGET_MIN session buy?

    The first iterations of a session are 5-10x slower than steady state: CUDA/cuDNN pick their
    kernels then, the dataloader workers start, and the first patches come off a cold, network
    mounted dataset.  Timing those (as this cell did) under-counts MAX_ITERS by the same factor:
    the advertised "~45 min per run" then finishes in 5 min having trained a fraction of the
    intended steps.  So: run PILOT_WARMUP iterations untimed, then time PILOT_TIMED of them.
    """
    model, opt = build()
    crit = make_loss(arm)
    use_skel = bool(SKEL_ROOT) and arm.startswith("sr")
    crit.w = crit.nominal            # the pilot times the term at its full strength
    ds = make_ds(arm, 0, (PILOT_WARMUP + PILOT_TIMED + 1) * BATCH)
    dl = torch.utils.data.DataLoader(ds, batch_size=BATCH, num_workers=NUM_WORKERS)
    model.train()
    t_start, n_timed = time.time(), 0
    for n, batch in enumerate(dl):
        if n == PILOT_WARMUP:
            # start timing ONLY once the pipeline is warm -- and synchronise, because CUDA ops
            # are enqueued asynchronously: without this the measured "iteration" is the time to
            # SUBMIT the work, not to run it (that read 47 it/s on a T4, ~10x too fast)
            if device.type == "cuda":
                torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
            t_start, n_timed = time.time(), 0
        x = batch[0].to(device)
        y = batch[1].to(device)
        dd = batch[2].to(device)
        sk = batch[3].to(device) if use_skel else None
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP, dtype=torch.float16):
            logits = model(x)
        loss = loss_forward(crit, logits, y, dd, sk, arm)
        if SCALER.is_enabled():              # the pilot takes a real step, so it needs the same
            SCALER.scale(loss).backward()    # protection as the training loop (never write a
            SCALER.unscale_(opt)             # non-finite gradient into the fp32 weights)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            SCALER.step(opt); SCALER.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
        if n == 0:
            # the first step is where a precision overflow shows up.  Checking it here costs
            # nothing and turns "45 minutes of nan" into a 20-second, self-explaining failure.
            _fin = dict(x=bool(torch.isfinite(x).all()), y=bool(torch.isfinite(y).all()),
                        dist=bool(torch.isfinite(dd).all()),
                        logits=bool(torch.isfinite(logits).all()),
                        loss=bool(torch.isfinite(loss)))
            print(f"first step: loss {float(loss):.6f} | finite: {_fin} | "
                  f"logits {float(logits.min()):+.2f}..{float(logits.max()):+.2f}")
            if not all(_fin.values()):
                with torch.no_grad():
                    _d = float(crit.dice(logits.float(), y.float()))
                    _b = float(bratsx.distance_weighted_bce(logits.float(), y.float(), dd.float()))
                raise RuntimeError(
                    f"the FIRST training step is not finite: {_fin}.\n"
                    f"  dice term (fp32) = {_d}, distance-weighted BCE (fp32) = {_b}\n"
                    f"  Most likely cause: the loss was computed under fp16 autocast.  MONAI's "
                    f"DiceLoss sums p^2 over a whole patch -- in fp16 that sum exceeds 65504 and "
                    f"becomes inf -> nan, and every weight is then nan for the rest of the run "
                    f"(the model predicts nothing; the validation score freezes at the "
                    f"'empty-region' value, e.g. 0.0556).  The fix in this build: keep the "
                    f"forward in fp16 but compute the loss in fp32 (logits.float()).  If the "
                    f"numbers above are finite, then the cache itself is suspect -- check that "
                    f"x/y/dist are finite (printed above).")
        if n >= PILOT_WARMUP:
            n_timed += 1
        if n >= PILOT_WARMUP + PILOT_TIMED - 1:
            break
    if device.type == "cuda":
        torch.cuda.synchronize()          # wait for the last enqueued iteration to finish
    dt = (time.time() - t_start) / max(n_timed, 1)
    peak = (torch.cuda.max_memory_allocated() / 1e9) if device.type == "cuda" else 0.0
    # floor of 500 applies to the TIME-derived budget; the two caps then bound the result
    # (the old order let the floor override MAX_ITERS_CAP)
    add = int(min(TARGET_ITERS, MAX_ITERS_CAP,
                  max(500, RUN_TARGET_MIN * 60 / dt * 0.8)))
    print(f"PILOT ({arm}): {1 / dt:.2f} it/s steady state | peak GPU mem {peak:.1f} GB | "
          f"~{dt * 1.25:.2f} s per iteration incl. validation")
    print(f"  (timed {n_timed} iterations after {PILOT_WARMUP} warm-up ones, with a CUDA "
          f"synchronise at both ends; the first iterations of a session are much slower and "
          f"would shrink the budget below)")
    print(f"  -> this session adds {add} iterations per run  (~{add * dt * 1.25 / 60:.0f} min)")
    if add < 2000:
        print(f"  !! {add} iterations is SHORT for training from scratch: expect a poor model "
              f"(rough WT Dice 0.3-0.6).  Raise RUN_TARGET_MIN, or run again later with "
              f"EXTEND=True to continue from last.pt -- iterations accumulate across sessions.")
    elif add < 6000:
        print(f"  note: {add} iterations is a modest budget; expect WT Dice ≈ 0.6-0.8 and watch "
              f"history.json: if the curve is still rising at the end, run more (EXTEND=True).")
    return add, dt, peak


MAX_ITERS_ADD, SEC_PER_IT, PEAK_GB = pilot(RUN_QUEUE[0][0] if RUN_QUEUE else "base")


# %% [CELL 7] train_one(arm, seed) -------------------------------------------
def train_one(arm, seed, add_iters=MAX_ITERS_ADD):
    rd = run_dir(arm, seed); os.makedirs(rd, exist_ok=True)
    if is_done(arm, seed):
        print(f"[{arm}/{seed}] already DONE -> skipped")
        return json.load(open(os.path.join(rd, "DONE")))
    _bad = _poisoned(rd)
    if _bad is not None and IGNORE_FAILED_RUNS:
        # move the dead attempt aside instead of resuming from it: resuming would load nan weights
        for _f in ("last.pt", "best.pt", "history.json", "train_log.csv", "val_last.csv", "DONE"):
            _p = os.path.join(rd, _f)
            if os.path.exists(_p):
                os.replace(_p, _p + ".failed")
        print(f"[{arm}/{seed}] the existing run records best={float(_bad.get('best', 0)):.4f} -- the "
              f"'predicts nothing' level (empty-prediction score, see train_log.csv for nan).  "
              f"Moving it aside as *.failed and training from scratch; set IGNORE_FAILED_RUNS=False "
              f"to keep such a run instead.")
    model, opt = build(seed)
    crit = make_loss(arm)
    use_skel = bool(SKEL_ROOT) and arm.startswith("sr")
    start_it, best = 0, -1.0
    ck = os.path.join(rd, "last.pt")
    if os.path.exists(ck):                                   # resume inside a run
        st = torch.load(ck, map_location=device, weights_only=False)
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"])
        start_it, best = st["iter"], st["best"]
        print(f"[{arm}/{seed}] resuming at it={start_it} best={best:.4f}")
        if _nan_weights(model):
            # a run that died of a non-finite gradient can leave a nan checkpoint behind.  It
            # cannot be resumed (rolling back to it would reload the same bad file), so it is
            # quarantined and the arm retrains from scratch -- in THIS session, no round trip.
            print(f"[{arm}/{seed}] last.pt holds nan weights (iteration {start_it}): it cannot be "
                  f"resumed, a rollback would reload the same file.  Moved aside as *.failed: "
                  f"{_quarantine(rd)}.  Training from scratch.", flush=True)
            model, opt = build(seed)
            crit = make_loss(arm)
            start_it, best = 0, -1.0

    max_iters = min(start_it + add_iters, MAX_ITERS_CAP)
    if max_iters <= start_it:
        print(f"[{arm}/{seed}] already at the iteration cap ({start_it}) -> nothing to add")
        return json.load(open(os.path.join(rd, "DONE")))
    if os.path.exists(os.path.join(rd, "DONE")):        # EXTEND case: this run continues
        os.remove(os.path.join(rd, "DONE"))
    # the dataset must cover the WHOLE run (see make_ds): one fresh sample per iteration
    ds = make_ds(arm, offset=start_it, length=(max_iters - start_it) * BATCH)
    dl = torch.utils.data.DataLoader(ds, batch_size=BATCH, num_workers=NUM_WORKERS)
    print(f"[{arm}/{seed}] training iterations {start_it} -> {max_iters}")
    val_every = max(200, max_iters // max(1, VAL_EVERY_ROUNDS))
    hist, t0, it, runs = [], time.time(), start_it, 0.0
    model.train()
    recover, skipped = 0, 0
    while it < max_iters:
        # the loader is (re)built here so that a rollback -- which moves `it` back -- also rewinds
        # the deterministic sample sequence instead of re-serving the patches it already saw
        ds = make_ds(arm, offset=it, length=max(1, (max_iters - it) * BATCH))
        dl = torch.utils.data.DataLoader(ds, batch_size=BATCH, num_workers=NUM_WORKERS)
        rolled_back = False
        for batch in dl:
            x = batch[0].to(device); y = batch[1].to(device); dd = batch[2].to(device)
            sk = batch[3].to(device) if use_skel else None
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP, dtype=torch.float16):
                logits = model(x)
                schedule_w(crit, it - start_it, 0)           # 0 at the first step, by construction
            # the loss reductions are fp32, the elementwise maths stays in the autocast dtype
            # (see loss_forward): a fp16 whole-patch sum overflows 65504 -> nan
            loss = loss_forward(crit, logits, y, dd, sk, arm)
            if it - start_it == 1:
                print(f"[{arm}/{seed}] first iteration: loss {float(loss):.6f} "
                      f"| inputs finite: "
                      f"{bool(torch.isfinite(x).all()) and bool(torch.isfinite(y).all())}",
                      flush=True)
            if not torch.isfinite(loss):
                if recover < MAX_RECOVER and os.path.exists(ck):
                    rolled_back = True
                    break
                with torch.no_grad():
                    _d = float(crit.dice(logits.float(), y.float()))
                    _b = float(bratsx.distance_weighted_bce(logits.float(), y.float(), dd.float()))
                    _wmax = max([float(q.abs().max()) for q in model.parameters()], default=0.0)
                raise RuntimeError(
                    f"the loss became non-finite ({float(loss)}) at iteration {it} -- "
                    f"dice term {_d}, distance-weighted BCE {_b}, inputs finite: "
                    f"{bool(torch.isfinite(x).all())}, largest weight {_wmax:.3g}.\n"
                    f"  Inputs fine + loss nan means the WEIGHTS are nan: a gradient wrote inf/nan "
                    f"into them.  The scaler skips such steps and MAX_RECOVER rolls a run back to "
                    f"its last checkpoint; both are exhausted here, so the run stops instead of "
                    f"training nan for hours.  Next things to try, in order: LR=5e-5, "
                    f"GRAD_CLIP=1.0, BATCH=1.")
            # ---- backward: scaler (skip non-finite gradients) + clipping ---------------------
            if SCALER.is_enabled():
                SCALER.scale(loss).backward()
                SCALER.unscale_(opt)                 # clipping must act on the unscaled gradients
                _gn = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                if not torch.isfinite(_gn):
                    skipped += 1
                SCALER.step(opt)                     # a no-op step when the gradients are bad
                SCALER.update()
            else:
                loss.backward()
                _gn = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                if torch.isfinite(_gn):
                    opt.step()
                else:                                # never write a bad gradient into the weights
                    skipped += 1
                    opt.zero_grad(set_to_none=True)
            runs += float(loss.detach()); it += 1
            if it % 50 == 0:
                _extra = f" | skipped steps {skipped}" if skipped else ""
                print(f"[{arm}/{seed}] it {it}/{max_iters} loss {runs/50:.4f} "
                      f"({(time.time()-t0)/max(1,it-start_it):.2f} s/it){_extra}", flush=True)
                with open(os.path.join(rd, "train_log.csv"), "a") as _fh:
                    _fh.write(f"{it},{runs/50:.6f}\n")
                runs = 0.0
            if it % CKPT_EVERY == 0 or it == max_iters:
                best_it = 0.0; per_case = {}
                if VAL_SUB:
                    best_it, per_case = validate(model, crit, VAL_SUB, max_iters, it)
                if per_case:
                    pd.DataFrame(per_case).T.to_csv(os.path.join(rd, "val_last.csv"))
                if best_it == best_it and best_it > best:       # not NaN and improved
                    best = best_it
                    torch.save({"model": model.state_dict(), "arm": arm, "seed": seed,
                                "iter": it, "best": best, "w_topology": float(crit.w)},
                               os.path.join(rd, "best.pt"))
                torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                            "crit": crit.state_dict(), "iter": it, "best": best}, ck)
                hist.append(dict(iter=it, best=best, w_topology=round(float(crit.w), 4)))
                json.dump(hist, open(os.path.join(rd, "history.json"), "w"), indent=1)
            if it >= max_iters:
                break
        if rolled_back:
            recover += 1
            del dl, ds                       # release the workers/dataset before building new ones
            gc.collect()
            _st = torch.load(ck, map_location=device, weights_only=False)
            model.load_state_dict(_st["model"]); opt.load_state_dict(_st["opt"])
            if _nan_weights(model):
                _moved = _quarantine(rd)
                raise RuntimeError(
                    f"the checkpoint at it={it} is itself poisoned (nan weights), so rolling back "
                    f"to it cannot help -- every forward pass from it is nan and the next rollback "
                    f"would reload the same file.  Its files were moved aside as *.failed "
                    f"({_moved}); re-run this arm and it will train from scratch.  Nothing else "
                    f"is affected and nothing needs deleting by hand.")
            for _g in opt.param_groups:
                _g["lr"] = max(float(_g["lr"]) * 0.5, 1e-6)
            it = int(_st.get("iter", start_it)); best = float(_st.get("best", best))
            runs = 0.0
            del _st
            print(f"[{arm}/{seed}] !! non-finite loss: rolled back to the checkpoint at it={it}, "
                  f"LR halved to {opt.param_groups[0]['lr']:.2e}, continuing "
                  f"(recovery {recover}/{MAX_RECOVER})", flush=True)
            continue
        if it >= max_iters:
            break
    secs = time.time() - t0
    payload = dict(arm=arm, seed=seed, iters=it, best=round(float(best), 4),
                   reached_target=bool(it >= min(TARGET_ITERS, MAX_ITERS_CAP)),
                   skipped_steps=int(skipped), recoveries=int(recover),
                   wall_min=round(secs / 60, 1), it_per_s=round((it - start_it) / max(secs, 1e-6), 3),
                   w_topology=round(float(crit.w), 4), max_iters=max_iters)
    json.dump(payload, open(os.path.join(rd, "DONE"), "w"), indent=1)
    print(f"[{arm}/{seed}] DONE at it={it} best={best:.4f} w_topo={payload['w_topology']} "
          f"({payload['wall_min']} min)")
    if use_skel:
        print(f"   structure-recall strength: ramped 0 -> {payload['w_topology']} "
              f"(nominal {crit.nominal}, warm-up {W_TOPOLOGY_WARMUP} iters).  Whether it earns "
              f"its place is decided in N2/N3 by the held-out metrics, not by the number here")
    return payload


# %% [CELL 8] RUN QUEUE ------------------------------------------------------
pending = [(a, s) for a, s in RUN_QUEUE if not is_done(a, s)]
print(f"session budget {TIME_BUDGET_H} h | runs pending: {len(pending)} of {len(RUN_QUEUE)}")
per_run_min = int(MAX_ITERS_ADD * max(SEC_PER_IT, 1e-6) / 60 * 1.25)
t_session = time.time()
print(f"each pending run adds up to {MAX_ITERS_ADD} iterations (~{per_run_min} min); "
      f"iterations already done are kept" + ("" if EXTEND else
      "  |  EXTEND=False: a run that is already DONE is skipped -- set EXTEND=True to continue it"))
finished, crashed = [], []
for arm, seed in pending:
    left_min = TIME_BUDGET_H * 60 - (time.time() - t_session) / 60
    if left_min < MIN_RUN_MIN:
        print(f"stopping: {left_min:.0f} min left (< MIN_RUN_MIN={MIN_RUN_MIN})")
        break
    if per_run_min > left_min:
        print(f"[{arm}/{seed}] needs ~{per_run_min} min, only {left_min:.0f} left -> "
              f"commit and continue in the next session (nothing is lost)")
        break
    try:
        finished.append(train_one(arm, seed))
    except Exception as _e:
        # A run that cannot continue must not take the session with it: the other arms are still
        # worth their GPU minutes, and this arm keeps last.pt so a later session can resume it.
        _msg = f"{type(_e).__name__}: {_e}"
        with open(os.path.join(run_dir(arm, seed), "CRASHED.txt"), "w") as _fh:
            _fh.write(_msg + "\n")
        crashed.append((arm, seed, _msg))
        print(f"[{arm}/{seed}] !! this run stopped: {_msg}\n"
              f"    the rest of the queue is unaffected; {arm}/{seed} keeps last.pt and resumes in "
              f"a later session (EXTEND=True).  See CRASHED.txt in its run directory.",
              flush=True)
if crashed:
    print("crashed this session: " + "; ".join(f"{a}/{s} ({m.splitlines()[0][:120]})"
                                               for a, s, m in crashed))
print(f"finished {len(finished)}/{len(pending)} pending runs this session; "
      f"DONE so far: {sum(is_done(a, s) for a, s in RUN_QUEUE)}/{len(RUN_QUEUE)}")


# %% [CELL 9] GPU ACCOUNTING + NEXT -----------------------------------------
_h = (time.time() - t_session) / 3600
print(f"""
GPU accounting (measured this session)
  one run          : ~{per_run_min} min   (at {1 / max(SEC_PER_IT, 1e-6):.2f} it/s, +{MAX_ITERS_ADD} iterations)
  runs per session : ~{max(1, int(TIME_BUDGET_H * 60 / max(per_run_min, 1)))}  (budget {TIME_BUDGET_H} h)
  this session     : {_h:.2f} GPU-hours   |  full queue of {len(RUN_QUEUE)} runs: ~
                     {len(RUN_QUEUE) * per_run_min / 60:.2f} GPU-hours
  weekly quota     : 30 GPU-hours -> {'comfortably inside' if len(RUN_QUEUE)*per_run_min/60 < 12 else 'check the account meter'}

NEXT
  1. Save Version -> Save & Run All (Commit) so the 9 h cap cannot kill the run.
  2. Output 'runs/' -> Create dataset (e.g. 'brats-runs-v1').
  3. New session: attach cache + skel + runs, set ADOPT_FROM = '/kaggle/input/brats-runs-v1',
     run the notebook again: unfinished runs resume from last.pt.
  3b. TO TRAIN LONGER than one session allows: set EXTEND=True and run again with the runs
     dataset attached -- each session adds another ~RUN_TARGET_MIN of iterations, up to
     TARGET_ITERS / MAX_ITERS_CAP, and the numbers accumulate.  Check history.json first: only
     continue if the validation curve is still rising.
  4. When the queue is done -> N2_eval_certify.

Reading history.json: 'w_topology' is the SCHEDULED strength of the structure-recall term
(0 -> nominal over W_TOPOLOGY_WARMUP iterations), not a learned quantity.  Whether the term
helps is answered in N2: the 'sr' arm must beat the control on the lesion-wise metrics and
must NOT regress on the easy strata -- if it does neither, report that instead of hiding it.
""")


library ready: 148 names (bratsx and safepp are aliases of this notebook's namespace)
GPU: Tesla T4 | mem 15.6 GB
autocast: True (fp16 forward, fp16 elementwise in the loss) | loss reductions in fp32 (a fp16 whole-patch sum overflows 65504 -> nan: that is what killed two runs)
cache: 1 shard(s) ['/kaggle/input/datasets/thinkbin/brats-cache-v2/cache_base']
skel:  1 shard(s) ['/kaggle/input/datasets/thinkbin/brats-cache-v2/skel']
split: train=295 val=37 (in-training subset 12)
skeleton patches are voxel-aligned with the targets (checked)


/tmp/ipykernel_23/825182908.py:2389: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(f"first step: loss {float(loss):.6f} | finite: {_fin} | "


first step: loss 1.947216 | finite: {'x': True, 'y': True, 'dist': True, 'logits': True, 'loss': True} | logits -3.36..+2.41
PILOT (base): 2.52 it/s steady state | peak GPU mem 2.4 GB | ~0.50 s per iteration incl. validation
  (timed 12 iterations after 3 warm-up ones, with a CUDA synchronise at both ends; the first iterations of a session are much slower and would shrink the budget below)
  -> this session adds 21795 iterations per run  (~180 min)
session budget 7.2 h | runs pending: 2 of 2
each pending run adds up to 21795 iterations (~179 min); iterations already done are kept
[base/42] training iterations 0 -> 21795
[base/42] first iteration: loss 1.698704 | inputs finite: True
[base/42] it 50/21795 loss 1.6218 (0.34 s/it) | skipped steps 1
[base/42] it 100/21795 loss 1.5194 (0.37 s/it) | skipped steps 1
[base/42] it 150/21795 loss 1.4595 (0.34 s/it) | skipped steps 1
[base/42] it 200/21795 loss 1.3909 (0.31 s/it) | skipped steps 1
[base/42] it 250/21795 loss 1.3277 (0.30 s/it) | s

In [3]:
!pip install monai